<a href="https://colab.research.google.com/github/lauracamilavargas-lang/predictive-maintenance-medical-imaging/blob/main/Modelo_para_protocolo_predictivo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Modelo de Mantenimiento Predictivo — Equipos biomédicos con principios de funcionamiento basado en radiación ionizantes en el hospital universitario fundación valle del lili.


# Bloque 1 — Pre-procesamiento de datos
El pre-procesamiento transforma el archivo Excel del hospital en un dataset limpio y estructurado listo para el entrenamiento de los modelos. El dataset de entrada corresponde a órdenes de mantenimiento correctivo registradas entre 2020 y 2025 para siete tipos de equipos de imagenología:

RX Fijo, RX Móvil, Arco C, TAC, Mamografía, Fluoroscopía y Angiografía.


El bloque ejecuta nueve pasos secuenciales: carga y consolidación del archivo Excel, estandarización de columnas, limpieza de variables clave y corrección de errores tipográficos, conversión de fechas y cálculo de horas de parada, cálculo del TBF (tiempo entre fallas consecutivas por equipo), construcción del target de urgencia para el Modelo A (Alta urgencia ≤15 d · Programable 16–90 d · Preventivo >90 d), generación de features temporales (fallos en 30 días, tendencia TBF, horas acumuladas), construcción del índice de criticidad por equipo (escala 1–5) y filtrado final para obtener df_modelo con los registros válidos para entrenamiento.

1.   Celda encargada de cargar el archivo Excel con la data tratada del hospital desde el computador local a Colab y se verifica que llegó correctamente mostrando sus dimensiones y nombres de columnas originales.

In [43]:
# ── CELDA 1 — CARGA DEL ARCHIVO ───────────────────────────────
import pandas as pd
import numpy as np
from google.colab import files

uploaded = files.upload()

if not uploaded:
    raise ValueError("No se subió ningún archivo. Vuelve a ejecutar la celda.")

nombre_archivo = list(uploaded.keys())[0]

if not nombre_archivo.endswith((".xlsx", ".xls")):
    raise ValueError(f"El archivo '{nombre_archivo}' no es un Excel válido.")

df = pd.read_excel(nombre_archivo)

if "TIPO DE EQUIPO" not in df.columns:
    raise ValueError("El archivo no contiene la columna 'TIPO DE EQUIPO'.")

# ── Resultados ─────────────────────────────────────────────────
print(f"Archivo        : {nombre_archivo}")
print(f"Shape          : {df.shape[0]} filas × {df.shape[1]} columnas")
print(f"Columnas       : {df.columns.tolist()}")
print(f"Tipos de equipo: {sorted(df['TIPO DE EQUIPO'].dropna().unique().tolist())}")
print(f"Nulos por col  :\n{df.isnull().sum()[df.isnull().sum() > 0].to_string()}")

Saving ANGIO,FLURO,FIJO,MOVIL,MAMO,ARCO, TC.xlsx to ANGIO,FLURO,FIJO,MOVIL,MAMO,ARCO, TC (1).xlsx
Archivo        : ANGIO,FLURO,FIJO,MOVIL,MAMO,ARCO, TC (1).xlsx
Shape          : 1230 filas × 23 columnas
Columnas       : ['ID', 'EQUIPO', 'ORDEN', 'TIPO DE EQUIPO', 'PARADA', 'Texto código motivo ', 'Horas de parada', 'fuera de servicio/gestion de la orden', 'Sistema en el que ocurrió la falla', 'Subsistema en el que ocurrió la falla', 'Componente que falló', 'componente que se cambio', 'solucion', 'tipo de solucion', 'Fecha puesta en marcha', 'Fecha falla', 'Motivo TBF', 'Cuenta con UPS', 'Preventivo', 'Sede', 'Ubicacion', 'Marca', 'Modelo']
Tipos de equipo: ['Sis Exploración Tomografía Computarizada', 'Sist Radiográf/Fluorosc Para Angiografía', 'Sistema radiográfico/fluoroscópico', 'Unidad Radiográfica Digital', 'Unidad Radiográfica Mamográfica', 'Unidad Radiográfica Móvil', 'Unidad Radiográfica/Fluoroscópica Móvil']
Nulos por col  :
PARADA                                   1109
Texto c

2.   Se normalizan todos los nombres de columnas eliminando tildes, espacios, barras y mayúsculas. Los nombres más largos se acortan a versiones manejables.



In [44]:
# ── CELDA 2 — ESTANDARIZAR NOMBRES DE COLUMNAS ────────────────
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("/", "_", regex=False)
    .str.replace("á", "a", regex=False)
    .str.replace("é", "e", regex=False)
    .str.replace("í", "i", regex=False)
    .str.replace("ó", "o", regex=False)
    .str.replace("ú", "u", regex=False)
    .str.replace("ñ", "n", regex=False)
    .str.replace("_+", "_", regex=True)   # colapsa dobles guiones bajos
    .str.strip("_")                        # elimina guiones al inicio/final
)

df = df.rename(columns={
    "tipo_de_equipo"                    : "tipo_equipo",
    "sistema_en_el_que_ocurrio_la_falla": "sistema_general",
    "subsistema_en_el_que_ocurrio_la_falla": "subsistema",
    "componente_que_fallo"              : "componente",
    "componente_que_se_cambio"          : "componente_cambiado",
    "texto_codigo_motivo"               : "tipo_falla",
    "tipo_de_solucion"                  : "tipo_solucion",
    "fuera_de_servicio_gestion_de_la_orden": "fuera_de_servicio",
    "horas_de_parada"                   : "horas_de_parada_num",
})

# ── Resultados ─────────────────────────────────────────────────
print(f"Columnas finales ({len(df.columns)}):")
print(df.columns.tolist())

Columnas finales (23):
['id', 'equipo', 'orden', 'tipo_equipo', 'parada', 'tipo_falla', 'horas_de_parada_num', 'fuera_de_servicio', 'sistema_general', 'subsistema', 'componente', 'componente_cambiado', 'solucion', 'tipo_solucion', 'fecha_puesta_en_marcha', 'fecha_falla', 'motivo_tbf', 'cuenta_con_ups', 'preventivo', 'sede', 'ubicacion', 'marca', 'modelo']


  2.1 Se normalizan los valores de las columnas de texto y se corrigen inconsistencias en sistema_general, consolidando variantes del mismo sistema y reclasificando registros mal asignados. El resultado son 11 clases limpias y consistentes

In [45]:
# ── CELDA 2b — LIMPIAR CONTENIDO DE COLUMNAS DE TEXTO ─────────
cols_texto = ["sistema_general", "subsistema", "componente",
              "tipo_falla", "tipo_solucion", "componente_cambiado"]

for col in cols_texto:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.lower()

# Fusionar variantes del mismo sistema
df["sistema_general"] = df["sistema_general"].replace({
    "interfaz de usuario"                          : "control e interfaz de usuario",
    "procesamiento, visualización y almacenamiento": "procesamiento y almacenamiento",
    "eléctrico fvl"                                : "eléctrico del equipo",
    "ti"                                           : "comunicaciones",
    "accesorios fvl"                               : "accesorios",
    "física médica"                                : "otro",
})

# Reclasificar registros mal asignados a "falla desconocida"
mask_proc = (
    (df["sistema_general"] == "falla desconocida") &
    (df["componente"].isin(["protocolos", "scoutview"]))
)
df.loc[mask_proc, "sistema_general"] = "procesamiento y almacenamiento"

mask_rx = (
    (df["sistema_general"] == "falla desconocida") &
    (df["componente"] == "articio en la imagen")
)
df.loc[mask_rx, "sistema_general"] = "generación y detección de rayos x"

# ── Resultados ─────────────────────────────────────────────────
print(f"Valores únicos sistema_general ({df['sistema_general'].nunique()}):")
print(df["sistema_general"].value_counts(dropna=False).to_string())
print(f"\nRegistros reclasificados desde 'falla desconocida': {mask_proc.sum() + mask_rx.sum()}")

Valores únicos sistema_general (11):
sistema_general
mecánico y de posicionamiento        197
generación y detección de rayos x    181
eléctrico del equipo                 167
comunicaciones                       155
procesamiento y almacenamiento       151
control e interfaz de usuario        149
falla desconocida                     84
accesorios                            61
usuario                               43
seguridad del paciente                41
otro                                   1

Registros reclasificados desde 'falla desconocida': 12


3. Se realiza la limpieza de horas de parada y se convierte horas_de_parada_num a formato numérico imputando con 0 los registros sin dato, correspondientes a intervenciones donde el equipo no salió de servicio.

In [46]:
# ── CELDA 3 — LIMPIEZA HORAS DE PARADA ────────────────────────
df["horas_de_parada_num"] = (
    df["horas_de_parada_num"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(",", ".", regex=False)
    .replace({"nan": "0", "n/a": "0", "na": "0", "-": "0", "": "0", "none": "0"})
    .pipe(pd.to_numeric, errors="coerce")
    .fillna(0)
)

# ── Resultados ─────────────────────────────────────────────────
print(f"Registros con horas_de_parada_num = 0 : {(df['horas_de_parada_num'] == 0).sum()}")
print(f"Registros con horas > 0              : {(df['horas_de_parada_num'] > 0).sum()}")
print(f"Máximo horas de parada               : {df['horas_de_parada_num'].max()}")
print(f"\nDistribución horas de parada:")
print(df["horas_de_parada_num"].describe().round(2))

Registros con horas_de_parada_num = 0 : 1103
Registros con horas > 0              : 127
Máximo horas de parada               : 1056.0

Distribución horas de parada:
count    1230.00
mean       12.71
std        77.19
min         0.00
25%         0.00
50%         0.00
75%         0.00
max      1056.00
Name: horas_de_parada_num, dtype: float64


4.    Se crea la variable binaria hubo_cambio_componente indicando si en la intervención se reemplazó algún componente físico. Los valores vacíos, "no aplica" y similares se interpretan como sin cambio (0).



In [47]:
# ── CELDA 4 — HUBO CAMBIO DE COMPONENTE ───────────────────────
df["componente_cambiado"] = (
    df["componente_cambiado"]
    .astype(str)
    .str.strip()
    .str.lower()
)

df["hubo_cambio_componente"] = np.where(
    df["componente_cambiado"].isin([
        "no aplica", "sin dato", "n/a",
        "na", "nan", "-", ""
    ]),
    0,
    1
)

# ── Resultados ─────────────────────────────────────────────────
total     = len(df)
con_cambio = df["hubo_cambio_componente"].sum()
print(f"Con cambio de componente    : {con_cambio} ({con_cambio/total*100:.1f}%)")
print(f"Sin cambio de componente    : {total - con_cambio} ({(total-con_cambio)/total*100:.1f}%)")
print(f"\nTop 10 componentes cambiados:")
print(df[df["hubo_cambio_componente"] == 1]["componente_cambiado"].value_counts().head(10).to_string())

Con cambio de componente    : 379 (30.8%)
Sin cambio de componente    : 851 (69.2%)

Top 10 componentes cambiados:
componente_cambiado
baterías interna               20
hand switch                    19
tubo de rayos x                16
carcasa del equipo             13
ruedas mecanicas               13
disco duro                     11
pines de carga detector         9
interruptor de pedal            9
mecanismo de parqueo (hook)     7
medidor de dosis (dap)          7


5. Construcción del índice de criticidad.

Se construye un índice de criticidad por combinación tipo_equipo × sistema_general en escala 1–5, ponderando proporción de tiempo fuera de servicio (40%), frecuencia relativa de fallos (40%) y horas de parada promedio (20%). Dos reglas de techo limitan el puntaje: combinaciones con menos de 15 fallas históricas no superan 3, y combinaciones con menos de 5 eventos fuera de servicio no superan 4. Con el dataset actual el máximo alcanzado es 4, ya que ninguna combinación cumple ambas condiciones simultáneamente.

In [48]:
# ── CELDA 5 — ÍNDICE DE CRITICIDAD ────────────────────────────
from sklearn.preprocessing import MinMaxScaler

# PASO 1 — Estado operacional
FUERA_DE_SERVICIO    = ["Equipo fuera de servicio"]
PARCIALMENTE_OPERATIVO = ["Equipo parcialmente operativo"]

def clasificar_estado(valor):
    if pd.isna(valor) or str(valor).strip() == "nan":
        return "operativo"
    elif valor in FUERA_DE_SERVICIO:
        return "fuera_de_servicio"
    elif valor in PARCIALMENTE_OPERATIVO:
        return "parcial"
    else:
        return "gestion"

df["estado_operacional"] = df["fuera_de_servicio"].apply(clasificar_estado)

# PASO 2 — Proporciones por tipo_equipo × sistema_general
df_crit = df[~df["sistema_general"].isin(["otro", "nan"])].copy()

def calcular_proporciones(grupo):
    total      = len(grupo)
    fuera      = (grupo["estado_operacional"] == "fuera_de_servicio").sum()
    horas_prom = grupo["horas_de_parada_num"].mean()
    return pd.Series({
        "total_fallas"        : total,
        "n_fuera_servicio"    : fuera,
        "prop_fuera_servicio" : fuera / total,
        "horas_parada_prom"   : horas_prom,
    })

base = (
    df_crit
    .groupby(["tipo_equipo", "sistema_general"])
    [["estado_operacional", "horas_de_parada_num"]]
    .apply(calcular_proporciones)
    .reset_index()
)

base["freq_relativa"] = base.groupby("tipo_equipo")["total_fallas"].transform(
    lambda x: x / x.sum()
)

scaler = MinMaxScaler()
base["horas_parada_norm"] = scaler.fit_transform(base[["horas_parada_prom"]])

# PASO 3 — Índice compuesto
PESO_FUERA = 0.4
PESO_FREQ  = 0.4
PESO_HORAS = 0.2

base["indice_raw"] = (
    base["prop_fuera_servicio"] * PESO_FUERA +
    base["freq_relativa"]       * PESO_FREQ  +
    base["horas_parada_norm"]   * PESO_HORAS
)

# PASO 4 — Normalizar 1-5 con reglas de techo
UMBRAL_FALLAS        = 15
UMBRAL_FUERA_CRITICO = 5

def normalizar_1_5(serie):
    min_v, max_v = serie.min(), serie.max()
    if max_v == min_v:
        return pd.Series([3] * len(serie), index=serie.index)
    return ((serie - min_v) / (max_v - min_v) * 4 + 1).round().astype(int)

base["criticidad"] = base.groupby("tipo_equipo")["indice_raw"].transform(normalizar_1_5)

mask_r1  = base["total_fallas"]     < UMBRAL_FALLAS
mask_r2  = base["n_fuera_servicio"] < UMBRAL_FUERA_CRITICO

base.loc[mask_r1, "criticidad"] = base.loc[mask_r1, "criticidad"].clip(upper=3)
base.loc[mask_r2, "criticidad"] = base.loc[mask_r2, "criticidad"].clip(upper=4)

# PASO 5 — Asignar al dataset
if "criticidad" in df.columns:
    df = df.drop(columns=["criticidad"])

df = df.merge(
    base[["tipo_equipo", "sistema_general", "criticidad"]],
    on=["tipo_equipo", "sistema_general"],
    how="left"
)

# ── Resultados ─────────────────────────────────────────────────
print(f"Distribución criticidad:")
print(df["criticidad"].value_counts(dropna=False).sort_index().to_string())
print(f"\nRegistros sin criticidad asignada : {df['criticidad'].isna().sum()}")
print(f"\nCriticidad por tipo de equipo:")
print(pd.crosstab(df["tipo_equipo"], df["criticidad"]))

Distribución criticidad:
criticidad
1.0     80
2.0    300
3.0    384
4.0    465
NaN      1

Registros sin criticidad asignada : 1

Criticidad por tipo de equipo:
criticidad                                1.0  2.0  3.0  4.0
tipo_equipo                                                 
Sis Exploración Tomografía Computarizada    2   35  161    0
Sist Radiográf/Fluorosc Para Angiografía   17   41   38    0
Sistema radiográfico/fluoroscópico         11   18   27    0
Unidad Radiográfica Digital                23   56   35  168
Unidad Radiográfica Mamográfica             5   19   48    0
Unidad Radiográfica Móvil                   9   50   38  260
Unidad Radiográfica/Fluoroscópica Móvil    13   81   37   37


6. Creación de las variables temporales: Se construyen cuatro features temporales por equipo: edad_dias (días desde puesta en marcha hasta el fallo), tbf_dias (tiempo entre fallos consecutivos del mismo equipo), horas_parada_acumuladas (horas fuera de servicio acumuladas históricamente) y fallos_30_dias (cantidad de fallos en los 30 días anteriores a cada evento). Los nulos en tbf_dias corresponden al primer registro de cada equipo.

In [49]:
# ── CELDA 6 — VARIABLES TEMPORALES ────────────────────────────
df["fecha_falla"]            = pd.to_datetime(df["fecha_falla"])
df["fecha_puesta_en_marcha"] = pd.to_datetime(df["fecha_puesta_en_marcha"])

df["edad_dias"] = (df["fecha_falla"] - df["fecha_puesta_en_marcha"]).dt.days

df = df.sort_values(["equipo", "fecha_falla"]).reset_index(drop=True)

df["tbf_dias"] = (
    df.groupby("equipo")["fecha_falla"]
    .diff()
    .dt.days
)

df["horas_parada_acumuladas"] = (
    df.groupby("equipo")["horas_de_parada_num"]
    .cumsum()
)

def fallos_ultimos_n(grupo, dias=30):
    fechas = pd.to_datetime(grupo["fecha_falla"])
    resultado = []
    for fecha in fechas:
        n = ((fechas < fecha) & (fechas >= fecha - pd.Timedelta(days=dias))).sum()
        resultado.append(n)
    return pd.Series(resultado, index=grupo.index)

df["fallos_30_dias"] = (
    df.groupby("equipo", group_keys=False)[["fecha_falla"]]
    .apply(fallos_ultimos_n)
)

def fallos_ultimos_90(grupo, dias=90):
    fechas = pd.to_datetime(grupo["fecha_falla"])
    resultado = []
    for fecha in fechas:
        n = ((fechas < fecha) & (fechas >= fecha - pd.Timedelta(days=dias))).sum()
        resultado.append(n)
    return pd.Series(resultado, index=grupo.index)

df["fallos_90_dias"] = (
    df.groupby("equipo", group_keys=False)[["fecha_falla"]]
    .apply(fallos_ultimos_90)
)

# ── Resultados ─────────────────────────────────────────────────
print(f"tbf_dias nulos          : {df['tbf_dias'].isna().sum()} (primer evento por equipo)")
print(f"edad_dias nulos         : {df['edad_dias'].isna().sum()}")
print(f"edad_dias negativos     : {(df['edad_dias'] < 0).sum()}")
print(f"\nResumen edad_dias:\n{df['edad_dias'].describe().round(2)}")
print(f"\nResumen tbf_dias:\n{df['tbf_dias'].describe().round(2)}")
print(f"\nResumen horas_parada_acumuladas:\n{df['horas_parada_acumuladas'].describe().round(2)}")
print(f"\nResumen fallos_30_dias:\n{df['fallos_30_dias'].describe().round(2)}")
print(f"\nResumen fallos_90_dias:\n{df['fallos_90_dias'].describe().round(2)}")

tbf_dias nulos          : 31 (primer evento por equipo)
edad_dias nulos         : 0
edad_dias negativos     : 0

Resumen edad_dias:
count    1230.00
mean     1914.52
std      1320.72
min        10.00
25%       773.25
50%      1738.00
75%      2873.50
max      5447.00
Name: edad_dias, dtype: float64

Resumen tbf_dias:
count    1199.00
mean       39.16
std        61.12
min         0.00
25%         7.00
50%        20.00
75%        44.00
max       631.00
Name: tbf_dias, dtype: float64

Resumen horas_parada_acumuladas:
count    1230.00
mean      343.11
std       496.77
min         0.00
25%         1.00
50%       119.00
75%       567.00
max      3627.00
Name: horas_parada_acumuladas, dtype: float64

Resumen fallos_30_dias:
count    1230.00
mean        1.16
std         1.38
min         0.00
25%         0.00
50%         1.00
75%         2.00
max         8.00
Name: fallos_30_dias, dtype: float64

Resumen fallos_90_dias:
count    1230.00
mean        3.40
std         3.03
min         0.00
25%    

7. Preparación final del dataset: Se calculan las features finales del pipeline: tbf_promedio_equipo usando solo fallos anteriores al actual para evitar data leakage, tendencia_tbf como diferencia logarítmica entre el TBF actual y el promedio histórico, tasa_fallo_equipo como ritmo de degradación relativo a la edad del equipo y dias_desde_ultimo_fallo como proxy de proximidad al evento anterior. Se construye df_modelo excluyendo los primeros registros de cada equipo sin TBF calculable. tbf_confiable se conserva para análisis pero no entra al modelo.

In [50]:
# CELDA 7 — PREPARACIÓN FINAL DEL DATASET

# TBF promedio histórico por equipo (solo fallos anteriores al actual)
df["tbf_promedio_equipo"] = df.groupby("equipo")["tbf_dias"].transform(
    lambda x: x.shift(1).expanding().mean()
)

# Tendencia del TBF respecto al promedio histórico por equipo
df["tendencia_tbf"] = df.groupby("equipo", group_keys=False).apply(
    lambda g: np.log1p(g["tbf_dias"].shift(1)) - np.log1p(
        g.groupby("equipo")["tbf_dias"].transform(
            lambda x: x.shift(1).expanding().mean()
        )
    )
).droplevel(0) if False else (
    np.log1p(df.groupby("equipo")["tbf_dias"].shift(1)) -
    np.log1p(df["tbf_promedio_equipo"])
)

# Corrección variantes motivo_tbf
df["motivo_tbf"] = df["motivo_tbf"].replace({
    "Falla simultánea distinto componente"  : "Falla simultáneas distintos componentes",
    "Falla simultánea distinto componente " : "Falla simultáneas distintos componentes",
}).fillna("normal")

# tbf_confiable — para análisis, no entra al modelo
df["tbf_confiable"] = np.where(
    df["motivo_tbf"] == "Falla simultáneas distintos componentes", 0, 1
)

# ── Construir df_modelo ────────────────────────────────────────
df_modelo = df[df["tbf_dias"].notna()].copy()

df_modelo["tbf_dias_log"] = np.log1p(df_modelo["tbf_dias"])

# Imputar nulos en features temporales
df_modelo["tbf_promedio_equipo"] = df_modelo["tbf_promedio_equipo"].fillna(
    df_modelo["tbf_promedio_equipo"].median()
)
df_modelo["tendencia_tbf"] = df_modelo["tendencia_tbf"].fillna(0)

# Features adicionales
df_modelo["fallos_acumulados"] = df_modelo.groupby("equipo").cumcount() + 1

df_modelo["tasa_fallo_equipo"] = (
    df_modelo["fallos_acumulados"] /
    df_modelo["edad_dias"].replace(0, np.nan)
).fillna(0)

df_modelo["dias_desde_ultimo_fallo"] = (
    df_modelo.groupby("equipo")["tbf_dias"]
    .shift(1)
    .fillna(0)
)

# ── Definición de features y targets ──────────────────────────
FEATURES_NUMERICAS = [
    "fallos_30_dias",
    "tbf_promedio_equipo",
    "tendencia_tbf",
    "criticidad",
    "edad_dias",
    "horas_parada_acumuladas",
    "tasa_fallo_equipo",
    "dias_desde_ultimo_fallo",
]

FEATURES_CATEGORICAS = [
    "tipo_solucion",
    "hubo_cambio_componente",
    "tipo_equipo",
]

FEATURES_TODAS  = FEATURES_NUMERICAS + FEATURES_CATEGORICAS
TARGET_PERIODO  = "periodo_fallo"
TARGET_SISTEMA  = "sistema_general_target"

# ── Resultados ─────────────────────────────────────────────────
print(f"df total   : {len(df)}")
print(f"df_modelo  : {len(df_modelo)}")
print(f"\nFeatures verificadas ({len(FEATURES_TODAS)}):")
for f in FEATURES_TODAS:
    ok    = "✓" if f in df_modelo.columns else "✗ FALTA"
    nulos = df_modelo[f].isna().sum() if f in df_modelo.columns else "N/A"
    print(f"  {ok} {f:<35} nulos: {nulos}")

df total   : 1230
df_modelo  : 1199

Features verificadas (11):
  ✓ fallos_30_dias                      nulos: 0
  ✓ tbf_promedio_equipo                 nulos: 0
  ✓ tendencia_tbf                       nulos: 0
  ✓ criticidad                          nulos: 1
  ✓ edad_dias                           nulos: 0
  ✓ horas_parada_acumuladas             nulos: 0
  ✓ tasa_fallo_equipo                   nulos: 0
  ✓ dias_desde_ultimo_fallo             nulos: 0
  ✓ tipo_solucion                       nulos: 0
  ✓ hubo_cambio_componente              nulos: 0
  ✓ tipo_equipo                         nulos: 0


# Bloque 2 — Análisis Exploratorio de Datos (EDA)
Antes de construir los modelos predictivos es fundamental entender la estructura y el comportamiento del dataset. El análisis exploratorio permite identificar patrones en las variables, detectar posibles problemas de balanceo entre clases y validar que las features construidas en el pre-procesamiento tienen sentido operacional antes de entregarlas al modelo.
En este bloque se analizan las distribuciones de las variables más relevantes, la correlación entre features, el balanceo de las clases objetivo para el Modelo A y el Modelo B, y se aplican técnicas de reducción de dimensionalidad — PCA y UMAP — para visualizar si los datos presentan separabilidad natural entre clases. Los resultados del EDA guiaron decisiones clave del pipeline como la selección de features, el manejo del desbalanceo de clases mediante compute_sample_weight y la definición de dos modelos XGBoost independientes.

8. Target periodo_fallo, se crea el target del Modelo A clasificando cada registro en una de tres categorías según el TBF: **Alta urgencia (0–15 días), Programable (16–90 días) y Preventivo (más de 90 días)**. Los nulos corresponden a registros sin TBF calculable que ya fueron excluidos en df_modelo.

In [51]:
# ── CELDA 8 — TARGET PERIODO_FALLO ────────────────────────────
RANGOS = [
    (0,  15,    "Alta urgencia"),
    (16, 90,    "Programable"),
    (91, 99999, "Preventivo"),
]

def clasificar_periodo(tbf):
    if pd.isna(tbf):
        return np.nan
    for lim_inf, lim_sup, nombre in RANGOS:
        if lim_inf <= tbf <= lim_sup:
            return nombre
    return np.nan

df_modelo["periodo_fallo"] = df_modelo["tbf_dias"].apply(clasificar_periodo)

# ── Resultados ─────────────────────────────────────────────────
total = df_modelo["periodo_fallo"].notna().sum()
print(f"Distribución target periodo_fallo (n={total}):")
for _, _, nombre in RANGOS:
    n   = (df_modelo["periodo_fallo"] == nombre).sum()
    pct = round(n / total * 100, 1)
    print(f"  {nombre:<20} {n:>5} registros  {pct}%")
print(f"\nNulos: {df_modelo['periodo_fallo'].isna().sum()}")

Distribución target periodo_fallo (n=1199):
  Alta urgencia          520 registros  43.4%
  Programable            555 registros  46.3%
  Preventivo             124 registros  10.3%

Nulos: 0


9. Target sistema_general. Se crea el target del Modelo B a partir de sistema_general. Las clases con menos de 30 registros se eliminan del dataset para garantizar suficiente representación durante el entrenamiento. El resultado son 10 clases que cubren todos los sistemas físicos de los equipos.

In [52]:
# ── CELDA 9 — TARGET SISTEMA_GENERAL ─────────────────────────
UMBRAL_REGISTROS_SISTEMA = 30

conteo_sis = df_modelo["sistema_general"].value_counts()
sistemas_minoritarios = conteo_sis[conteo_sis < UMBRAL_REGISTROS_SISTEMA].index.tolist()

df_modelo["sistema_general_target"] = df_modelo["sistema_general"].apply(
    lambda x: "otro" if x in sistemas_minoritarios else x
).str.lower()

# Eliminar registros con sistema_general_target == "otro"
df_modelo = df_modelo[df_modelo["sistema_general_target"] != "otro"].copy()

TARGET_SISTEMA = "sistema_general_target"

# ── Resultados ─────────────────────────────────────────────────
print(f"Umbral mínimo por clase    : {UMBRAL_REGISTROS_SISTEMA} registros")
print(f"Clases eliminadas          : {sistemas_minoritarios}")
print(f"Registros tras limpieza    : {len(df_modelo)}")
print(f"\nDistribución sistema_general_target ({df_modelo[TARGET_SISTEMA].nunique()} clases):")
conteo_final = df_modelo[TARGET_SISTEMA].value_counts()
for sistema, n in conteo_final.items():
    pct = round(n / len(df_modelo) * 100, 1)
    print(f"  {sistema:<40} {n:>5}  {pct}%")
print(f"\nTARGET_PERIODO : {TARGET_PERIODO}")
print(f"TARGET_SISTEMA : {TARGET_SISTEMA}")

Umbral mínimo por clase    : 30 registros
Clases eliminadas          : ['otro']
Registros tras limpieza    : 1198

Distribución sistema_general_target (10 clases):
  mecánico y de posicionamiento              192  16.0%
  generación y detección de rayos x          178  14.9%
  eléctrico del equipo                       163  13.6%
  comunicaciones                             151  12.6%
  procesamiento y almacenamiento             149  12.4%
  control e interfaz de usuario              142  11.9%
  falla desconocida                           82  6.8%
  accesorios                                  61  5.1%
  usuario                                     42  3.5%
  seguridad del paciente                      38  3.2%

TARGET_PERIODO : periodo_fallo
TARGET_SISTEMA : sistema_general_target


10. Resume por equipo. Tabla interactiva con métricas de confiabilidad por equipo: TBF promedio, mediana y moda, rango de TBF, horas de parada acumuladas, porcentaje de registros no confiables y máximo de fallos en 30 días. El semáforo de confiabilidad indica qué equipos tienen mayor proporción de TBF no confiables por fallas simultáneas.

In [53]:
# ── CELDA 10 — RESUMEN POR EQUIPO ─────────────────────────────
from IPython.display import display, HTML

resumen = df.groupby(["equipo", "tipo_equipo"]).agg(
    total_registros    = ("id",                      "count"),
    tbf_promedio       = ("tbf_dias",                "mean"),
    tbf_mediana        = ("tbf_dias",                "median"),
    tbf_moda           = ("tbf_dias",                lambda x: x.mode()[0] if len(x.mode()) > 0 else 0),
    tbf_min            = ("tbf_dias",                "min"),
    tbf_max            = ("tbf_dias",                "max"),
    edad_promedio      = ("edad_dias",               "mean"),
    horas_acum_total   = ("horas_parada_acumuladas",  "max"),
    fallos_30_max      = ("fallos_30_dias",           "max"),
    no_confiables      = ("tbf_confiable",            lambda x: (x == 0).sum()),
).reset_index()

resumen["tbf_promedio"]     = resumen["tbf_promedio"].round(1)
resumen["tbf_mediana"]      = resumen["tbf_mediana"].round(1)
resumen["tbf_moda"]         = resumen["tbf_moda"].round(1)
resumen["edad_promedio"]    = (resumen["edad_promedio"] / 365).round(1)
resumen["pct_no_confiable"] = (resumen["no_confiables"] / resumen["total_registros"] * 100).round(1)

TIPO_COLOR = {
    "Unidad Radiográfica Digital"              : {"bg":"#2a1a2a","acc":"#7F77DD","short":"RX Fijo"},
    "Unidad Radiográfica Mamográfica"          : {"bg":"#2a1a1a","acc":"#D85A30","short":"Mamografía"},
    "Unidad Radiográfica Móvil"                : {"bg":"#1a2626","acc":"#1D9E75","short":"RX Móvil"},
    "Unidad Radiográfica/Fluoroscópica Móvil"  : {"bg":"#26221a","acc":"#BA7517","short":"Arco C"},
    "Sist Radiográf/Fluorosc Para Angiografía" : {"bg":"#1a2636","acc":"#378ADD","short":"Angiografía"},
    "Sistema radiográfico/fluoroscópico"       : {"bg":"#1a2a1a","acc":"#639922","short":"Fluoroscopía"},
    "Sis Exploración Tomografía Computarizada" : {"bg":"#26261a","acc":"#E24B4A","short":"TAC"},
}

def barra(valor, maximo, color, ancho=120):
    pct = min(100, int(valor / maximo * 100)) if maximo > 0 else 0
    px  = int(ancho * pct / 100)
    return (
        "<div style='display:flex;align-items:center;gap:6px'>"
        "<div style='width:" + str(ancho) + "px;background:#2a2a2a;border-radius:3px;height:6px'>"
        "<div style='width:" + str(px) + "px;background:" + color + ";border-radius:3px;height:6px'></div>"
        "</div>"
        "<span style='font-size:11px;color:#aaa;min-width:36px'>" + str(valor) + "</span>"
        "</div>"
    )

def badge_confiable(pct):
    if pct >= 30:
        return "<span style='background:#4a1a1a;color:#f09595;padding:2px 8px;border-radius:99px;font-size:11px;font-weight:700'>⚠️ " + str(pct) + "%</span>"
    elif pct >= 10:
        return "<span style='background:#3a2e10;color:#EF9F27;padding:2px 8px;border-radius:99px;font-size:11px'>⚡ " + str(pct) + "%</span>"
    else:
        return "<span style='background:#1a2e1a;color:#97c459;padding:2px 8px;border-radius:99px;font-size:11px'>✅ " + str(pct) + "%</span>"

def celda_tbf(mediana, moda, color):
    return (
        "<div style='display:flex;flex-direction:column;gap:4px'>"
        "<span style='font-size:14px;font-weight:700;color:" + color + "'>" + str(mediana) + " d</span>"
        "<div style='display:flex;align-items:center;gap:5px'>"
        "<span style='font-size:9px;color:#555;text-transform:uppercase;letter-spacing:0.05em'>moda</span>"
        "<span style='background:#1e1e1e;color:#888;font-size:11px;padding:1px 7px;border-radius:99px;border:1px solid #333'>" + str(moda) + " d</span>"
        "</div>"
        "</div>"
    )

max_horas     = resumen["horas_acum_total"].max()
max_registros = resumen["total_registros"].max()

estilos = (
    "<style>"
    ".pw{font-family:Arial,sans-serif;color:#e0ddd6;padding:8px 0}"
    ".pw-tipo{margin:20px 0 0;padding:10px 14px;border-radius:10px 10px 0 0;font-size:13px;font-weight:700;display:flex;align-items:center;gap:8px}"
    ".pw-table{width:100%;border-collapse:collapse;border:1px solid #333;border-top:none;margin-bottom:4px}"
    ".pw-table th{background:#1e1e1e;padding:8px 12px;font-size:10px;color:#666;font-weight:600;text-align:left;border-bottom:1px solid #333;text-transform:uppercase;letter-spacing:.05em}"
    ".pw-table td{padding:10px 12px;font-size:12px;border-bottom:1px solid #1e1e1e;background:#141414;vertical-align:middle}"
    ".pw-table tr:last-child td{border-bottom:none}"
    ".pw-table tr:hover td{background:#1c1c1c}"
    ".eq-id{font-size:15px;font-weight:700;color:#f0ede8;display:block}"
    ".eq-edad{font-size:11px;color:#666;margin-top:2px}"
    ".tbf-rng{font-size:10px;color:#555;margin-top:2px}"
    ".legend{display:flex;gap:20px;flex-wrap:wrap;margin-bottom:16px;font-size:11px;color:#666}"
    ".legend-item{display:flex;align-items:center;gap:6px}"
    ".leg-dot{width:8px;height:8px;border-radius:50%}"
    "</style>"
)

leyenda = (
    "<div class='legend'>"
    "<div class='legend-item'><div class='leg-dot' style='background:#f09595'></div> >= 30% TBF no confiables</div>"
    "<div class='legend-item'><div class='leg-dot' style='background:#EF9F27'></div> 10-29% TBF no confiables</div>"
    "<div class='legend-item'><div class='leg-dot' style='background:#97c459'></div> < 10% TBF no confiables</div>"
    "</div>"
)

html = estilos + "<div class='pw'>" + leyenda

for tipo, grupo in resumen.groupby("tipo_equipo"):
    cfg  = TIPO_COLOR.get(tipo, {"bg":"#1a1a1a","acc":"#888","short":tipo[:8]})
    n_eq = len(grupo)

    html += (
        "<div class='pw-tipo' style='background:" + cfg["bg"] + ";border-left:4px solid " + cfg["acc"] + "'>"
        + cfg["short"] + " &nbsp;<span style='font-weight:400;color:#888;font-size:12px'>— " + tipo + "</span>"
        "<span style='margin-left:auto;font-size:11px;color:#555'>" + str(n_eq) + " equipo" + ("s" if n_eq > 1 else "") + "</span>"
        "</div>"
        "<table class='pw-table'><thead><tr>"
        "<th>Equipo</th>"
        "<th>Registros</th>"
        "<th>TBF promedio</th>"
        "<th>TBF mediana / moda</th>"
        "<th>Rango TBF</th>"
        "<th>% No confiable</th>"
        "<th>Horas parada acum.</th>"
        "<th>Max fallos 30d</th>"
        "</tr></thead><tbody>"
    )

    for _, row in grupo.sort_values("tbf_mediana").iterrows():
        tbf_color = cfg["acc"]
        tbf_min   = "—" if pd.isna(row["tbf_min"]) else str(int(row["tbf_min"]))
        tbf_max   = "—" if pd.isna(row["tbf_max"]) else str(int(row["tbf_max"]))
        tbf_prom  = "—" if pd.isna(row["tbf_promedio"]) else str(row["tbf_promedio"])
        tbf_med   = "—" if pd.isna(row["tbf_mediana"])  else str(row["tbf_mediana"])
        tbf_moda  = "—" if pd.isna(row["tbf_moda"])     else str(row["tbf_moda"])

        html += (
            "<tr>"
            "<td><span class='eq-id'>" + str(int(row["equipo"])) + "</span>"
            "<span class='eq-edad'>🕐 " + str(row["edad_promedio"]) + " años promedio</span></td>"
            "<td>" + barra(row["total_registros"], max_registros, cfg["acc"]) + "</td>"
            "<td><span style='font-size:14px;font-weight:700;color:" + tbf_color + "'>" + tbf_prom + " d</span></td>"
            "<td>" + celda_tbf(tbf_med, tbf_moda, tbf_color) + "</td>"
            "<td><span class='tbf-rng'>" + tbf_min + " → " + tbf_max + " dias</span></td>"
            "<td>" + badge_confiable(row["pct_no_confiable"]) + "</td>"
            "<td>" + barra(row["horas_acum_total"], max_horas, "#E24B4A") + "</td>"
            "<td style='text-align:center;font-weight:700;color:" + tbf_color + "'>" + str(int(row["fallos_30_max"])) + "</td>"
            "</tr>"
        )

    html += "</tbody></table>"

html += "</div>"
display(HTML(html))

# ── Resultados ─────────────────────────────────────────────────
print(f"Equipos analizados  : {resumen['equipo'].nunique()}")
print(f"Tipos de equipo     : {resumen['tipo_equipo'].nunique()}")
print(f"TBF promedio global : {resumen['tbf_promedio'].mean():.1f} días")
print(f"Max horas parada    : {resumen['horas_acum_total'].max():.0f} h")

Equipo,Registros,TBF promedio,TBF mediana / moda,Rango TBF,% No confiable,Horas parada acum.,Max fallos 30d
203025🕐 8.2 años promedio,103,20.0 d,14.0 dmoda0.0 d,0 → 94 dias,✅ 4.9%,1895.0,5
203846🕐 5.0 años promedio,40,51.7 d,22.0 dmoda22.0 d,0 → 412 dias,✅ 0.0%,29.0,3
200790🕐 12.5 años promedio,40,51.9 d,27.0 dmoda10.0 d,0 → 290 dias,✅ 7.5%,837.0,3
206278🕐 3.0 años promedio,16,102.1 d,90.0 dmoda0.0 d,0 → 296 dias,✅ 6.2%,8.0,1
Equipo,Registros,TBF promedio,TBF mediana / moda,Rango TBF,% No confiable,Horas parada acum.,Max fallos 30d
208677🕐 0.7 años promedio,21,41.9 d,16.0 dmoda6.0 d,0 → 312 dias,✅ 0.0%,191.0,4
207343🕐 1.3 años promedio,30,33.1 d,24.0 dmoda0.0 d,0 → 130 dias,✅ 0.0%,71.0,6
207342🕐 1.1 años promedio,26,31.6 d,26.0 dmoda0.0 d,0 → 84 dias,✅ 0.0%,0.0,2
206796🕐 7.9 años promedio,19,110.7 d,66.5 dmoda0.0 d,0 → 447 dias,✅ 5.3%,531.0,1
Equipo,Registros,TBF promedio,TBF mediana / moda,Rango TBF,% No confiable,Horas parada acum.,Max fallos 30d


Equipos analizados  : 31
Tipos de equipo     : 7
TBF promedio global : 62.3 días
Max horas parada    : 3627 h


11. Varibles de entrada Modelo A. Tabla de las 9 features del Modelo A clasificadas por tipo e importancia esperada. Las variables de frecuencia reciente e histórica (fallos_30_dias, tbf_promedio_equipo) son las de mayor relevancia esperada por capturar directamente el ritmo de fallos del equipo.

In [54]:
# ── CELDA 11 — VARIABLES DE ENTRADA MODELO A ──────────────────
import plotly.graph_objects as go

variables = pd.DataFrame([
    ["fallos_30_dias",          "Numérica",   "Fallos del equipo en los últimos 30 días",           "Alta"],
    ["tbf_promedio_equipo",     "Numérica",   "TBF promedio histórico del equipo hasta ese evento", "Alta"],
    ["tendencia_tbf",           "Numérica",   "Desviación del TBF actual respecto al histórico",    "Media"],
    ["criticidad",              "Numérica",   "Índice compuesto de gravedad (1-5)",                 "Media"],
    ["edad_dias",               "Numérica",   "Días instalado al momento del fallo",                "Media"],
    ["horas_parada_acumuladas", "Numérica",   "Total horas fuera de servicio acumuladas",           "Media"],
    ["tasa_fallo_equipo",       "Numérica",   "Fallos acumulados dividido edad en días",            "Media"],
    ["dias_desde_ultimo_fallo", "Numérica",   "Días transcurridos desde el fallo anterior",         "Media"],
    ["tipo_equipo",             "Categórica", "Tipo de equipo médico",                              "Media"],
], columns=["Variable", "Tipo", "Descripción", "Importancia esperada"])

color_tipo = {"Numérica": "#1a3a5c", "Categórica": "#1a3a2a"}
color_imp  = {"Alta": "#1a4a2a", "Media": "#3a3a1a"}

fig = go.Figure(data=[go.Table(
    columnwidth=[220, 120, 380, 180],
    header=dict(
        values=["<b>Variable</b>", "<b>Tipo</b>", "<b>Descripción</b>", "<b>Importancia esperada</b>"],
        fill_color="#1F4E79",
        font=dict(color="white", size=13, family="Arial"),
        align="left",
        height=36
    ),
    cells=dict(
        values=[
            variables["Variable"].tolist(),
            variables["Tipo"].tolist(),
            variables["Descripción"].tolist(),
            variables["Importancia esperada"].tolist(),
        ],
        fill_color=[
            ["#0d1117"] * len(variables),
            [color_tipo.get(t, "#1a1a1a") for t in variables["Tipo"]],
            ["#0d1117"] * len(variables),
            [color_imp.get(i, "#1a1a1a")  for i in variables["Importancia esperada"]],
        ],
        font=dict(color=["#e0ddd6", "white", "#cccccc", "white"], size=12, family="Arial"),
        align="left",
        height=38
    )
)])

fig.update_layout(
    title=dict(
        text="Variables de entrada — Modelo A (periodo_fallo) — 3 clases: Alta urgencia / Programable / Preventivo",
        font=dict(size=14, color="#e0ddd6", family="Arial"),
        x=0.01
    ),
    margin=dict(t=50, b=10, l=0, r=0),
    paper_bgcolor="#0d1117",
    height=400
)
fig.show()

# ── Resultados ─────────────────────────────────────────────────
print(f"Total variables Modelo A : {len(variables)}")
print(f"Numéricas                : {(variables['Tipo'] == 'Numérica').sum()}")
print(f"Categóricas              : {(variables['Tipo'] == 'Categórica').sum()}")
print(f"Target                   : {TARGET_PERIODO} — 3 clases")

Total variables Modelo A : 9
Numéricas                : 8
Categóricas              : 1
Target                   : periodo_fallo — 3 clases


12. Distibución del target periodo_fallo. Tres visualizaciones complementarias: dona con el total de registros al centro para las proporciones globales, barras horizontales apiladas para comparar volumen por tipo de equipo, y barras horizontales al 100% para revelar la variación en comportamiento de fallo entre equipos. El panel de proporciones evidencia que Arco C concentra el 35% de sus registros en Preventivo mientras RX Móvil tiene menos del 5%, justificando el uso de **compute_sample_weight** en el entrenamiento.

In [55]:
# ── CELDA 12 — DISTRIBUCIÓN TARGET PERIODO_FALLO ──────────────
import plotly.graph_objects as go
from plotly.subplots import make_subplots

NOMBRES_CORTOS = {
    "Sis Exploración Tomografía Computarizada" : "TAC",
    "Sistema radiográfico/fluoroscópico"        : "Fluoro",
    "Unidad Radiográfica Digital"               : "RX Fijo",
    "Unidad Radiográfica Mamográfica"           : "Mamografía",
    "Unidad Radiográfica Móvil"                 : "RX Móvil",
    "Unidad Radiográfica/Fluoroscópica Móvil"   : "Arco C",
    "Sist Radiográf/Fluorosc Para Angiografía"  : "Angiografía",
}

df_modelo["tipo_corto"] = df_modelo["tipo_equipo"].map(NOMBRES_CORTOS)

CLASES   = ["Alta urgencia", "Programable", "Preventivo"]
COLORES  = ["#E24B4A", "#378ADD", "#1D9E75"]
TIPOS    = ["Mamografía", "Fluoro", "Angiografía", "Arco C", "TAC", "RX Fijo", "RX Móvil"]

conteo_global = {c: (df_modelo["periodo_fallo"] == c).sum() for c in CLASES}
total         = sum(conteo_global.values())

conteo_tipo = {}
for tipo in TIPOS:
    sub = df_modelo[df_modelo["tipo_corto"] == tipo]
    conteo_tipo[tipo] = {c: (sub["periodo_fallo"] == c).sum() for c in CLASES}

fig = make_subplots(
    rows=2, cols=2,
    specs=[[{"type": "domain"}, {"type": "bar"}],
           [{"type": "bar", "colspan": 2}, None]],
    subplot_titles=(
        "Distribución global",
        "Volumen por tipo de equipo",
        "Proporción por tipo de equipo — variación en comportamiento de fallo entre equipos",
    ),
    vertical_spacing=0.14,
    horizontal_spacing=0.08,
)

# Panel 1 — Dona
fig.add_trace(go.Pie(
    labels=CLASES,
    values=[conteo_global[c] for c in CLASES],
    marker_colors=COLORES,
    hole=0.62,
    textinfo="none",
    hovertemplate="<b>%{label}</b><br>%{value} registros (%{percent})<extra></extra>",
), row=1, col=1)

# Anotación central
fig.add_annotation(
    text=f"<b>{total:,}</b><br><span style='font-size:11px'>registros</span>",
    x=0.18, y=0.75, xref="paper", yref="paper",
    showarrow=False,
    font=dict(size=16, color="#e0ddd6", family="Arial"),
    align="center",
)

# Panel 2 — Barras horizontales apiladas (volumen)
for clase, color in zip(CLASES, COLORES):
    fig.add_trace(go.Bar(
        name=clase,
        y=TIPOS,
        x=[conteo_tipo[t][clase] for t in TIPOS],
        orientation="h",
        marker_color=color,
        marker_line_width=0,
        hovertemplate="<b>%{y}</b><br>" + clase + ": %{x}<extra></extra>",
        showlegend=True,
    ), row=1, col=2)

# Panel 3 — Barras horizontales 100% apiladas (proporción)
for clase, color in zip(CLASES, COLORES):
    fig.add_trace(go.Bar(
        name=clase,
        y=TIPOS,
        x=[round(conteo_tipo[t][clase] /
                 sum(conteo_tipo[t].values()) * 100, 1) for t in TIPOS],
        orientation="h",
        marker_color=color,
        marker_line_width=0,
        hovertemplate="<b>%{y}</b><br>" + clase + ": %{x}%<extra></extra>",
        showlegend=False,
    ), row=2, col=1)

fig.update_layout(
    title=dict(
        text="Clasificación del target — Periodo de fallo (3 clases operacionales)",
        font=dict(size=15, color="#e0ddd6", family="Arial"),
        x=0.01
    ),
    barmode="stack",
    paper_bgcolor="#0d1117",
    plot_bgcolor="#0d1117",
    font=dict(color="#e0ddd6", family="Arial", size=11),
    height=620,
    margin=dict(t=80, b=40, l=100, r=40),
    legend=dict(
        bgcolor="#1a1a1a",
        bordercolor="#333333",
        borderwidth=1,
        font=dict(size=11),
        orientation="h",
        y=-0.05,
        x=0.3,
    )
)

fig.update_xaxes(gridcolor="#222222", tickfont=dict(size=10), row=1, col=2)
fig.update_xaxes(gridcolor="#222222", tickfont=dict(size=10),
                 ticksuffix="%", range=[0, 100], row=2, col=1)
fig.update_yaxes(gridcolor="#222222", tickfont=dict(size=10), row=1, col=2)
fig.update_yaxes(gridcolor="#222222", tickfont=dict(size=10), row=2, col=1)
fig.show()

# ── Resultados ─────────────────────────────────────────────────
print(f"Distribución global (n={total}):")
for clase, color in zip(CLASES, COLORES):
    n   = conteo_global[clase]
    pct = round(n / total * 100, 1)
    alerta = "⚠ subrepresentada — justifica compute_sample_weight" if pct < 15 else "✓ balance aceptable"
    print(f"  {clase:<20} {n:>5} registros  {pct:>5}%  {alerta}")

Distribución global (n=1198):
  Alta urgencia          520 registros   43.4%  ✓ balance aceptable
  Programable            555 registros   46.3%  ✓ balance aceptable
  Preventivo             123 registros   10.3%  ⚠ subrepresentada — justifica compute_sample_weight


13. Distribución del TBF.
Cuatro paneles complementarios: histograma apilado por zona de clasificación que evidencia el sesgo a la derecha, TBF mediano por equipo para comparar comportamiento entre tipos, boxplot con eje limitado a 200 días para lectura sin distorsión por outliers, y curva de supervivencia empírica con los cortes en 15 y 90 días que definen las tres clases del Modelo A. La alta asimetría del TBF **justifica usar clasificación en lugar de regresión.**

In [56]:
# ── CELDA 13 — DISTRIBUCIÓN DEL TARGET TBF_DIAS ───────────────
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

NOMBRES_CORTOS = {
    "Sis Exploración Tomografía Computarizada" : "TAC",
    "Sistema radiográfico/fluoroscópico"        : "Fluoro",
    "Unidad Radiográfica Digital"               : "RX Fijo",
    "Unidad Radiográfica Mamográfica"           : "Mamografía",
    "Unidad Radiográfica Móvil"                 : "RX Móvil",
    "Unidad Radiográfica/Fluoroscópica Móvil"   : "Arco C",
    "Sist Radiográf/Fluorosc Para Angiografía"  : "Angiografía",
}

df_modelo["tipo_corto"] = df_modelo["tipo_equipo"].map(NOMBRES_CORTOS)

COLORES = {"Alta urgencia": "#E24B4A", "Programable": "#378ADD", "Preventivo": "#1D9E75"}

def hex_a_rgba(hex_color, alpha=0.25):
    h = hex_color.lstrip("#")
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return f"rgba({r},{g},{b},{alpha})"

def color_mediana(v):
    if v <= 15: return "#E24B4A"
    if v <= 90: return "#378ADD"
    return "#1D9E75"

# ── Datos histograma ──────────────────────────────────────────
bins     = list(range(0, 310, 5))
hist_alta = np.histogram(df_modelo[df_modelo["periodo_fallo"] == "Alta urgencia"]["tbf_dias"].dropna(), bins=bins)[0]
hist_prog = np.histogram(df_modelo[df_modelo["periodo_fallo"] == "Programable"]["tbf_dias"].dropna(),   bins=bins)[0]
hist_prev = np.histogram(df_modelo[df_modelo["periodo_fallo"] == "Preventivo"]["tbf_dias"].dropna(),    bins=bins)[0]
bin_mids  = [(bins[i] + bins[i+1]) / 2 for i in range(len(bins)-1)]

# ── Datos TBF mediano por equipo ──────────────────────────────
medianas_equipo = (
    df_modelo.groupby("tipo_corto")["tbf_dias"]
    .median()
    .sort_values()
    .reset_index()
)
medianas_equipo.columns = ["tipo", "mediana"]

# ── Datos boxplot ─────────────────────────────────────────────
df_box    = df_modelo[df_modelo["tbf_dias"] <= 200].copy()
tipos_box = (
    df_box.groupby("tipo_corto")["tbf_dias"]
    .median()
    .sort_values()
    .index.tolist()
)

# ── Curva de supervivencia ────────────────────────────────────
tbf_sorted = np.sort(df_modelo["tbf_dias"].dropna().values)
survival   = 1 - np.arange(1, len(tbf_sorted) + 1) / len(tbf_sorted)

# ── Figura ────────────────────────────────────────────────────
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "Distribución de TBF — zonas por clase",
        "TBF mediano por tipo de equipo",
        "Boxplot por tipo de equipo (eje limitado a 200d)",
        "Curva de supervivencia empírica — P(TBF > t)",
    ),
    vertical_spacing=0.18,
    horizontal_spacing=0.12,
)

# Panel 1 — histograma apilado por zona
for clase, hist in [("Alta urgencia", hist_alta),
                    ("Programable",   hist_prog),
                    ("Preventivo",    hist_prev)]:
    fig.add_trace(go.Bar(
        x=bin_mids, y=hist,
        name=clase,
        marker_color=COLORES[clase],
        marker_line_width=0,
        showlegend=True,
    ), row=1, col=1)

# Panel 2 — TBF mediano horizontal
fig.add_trace(go.Bar(
    x=medianas_equipo["mediana"],
    y=medianas_equipo["tipo"],
    orientation="h",
    marker_color=[color_mediana(v) for v in medianas_equipo["mediana"]],
    marker_line_width=0,
    text=[f"{v:.0f}d" for v in medianas_equipo["mediana"]],
    textposition="outside",
    textfont=dict(size=11, color="#aaaaaa"),
    showlegend=False,
), row=1, col=2)

# Panel 3 — boxplot por equipo
for tipo in tipos_box:
    sub   = df_box[df_box["tipo_corto"] == tipo]["tbf_dias"]
    med   = sub.median()
    color = color_mediana(med)
    fig.add_trace(go.Box(
        y=sub,
        name=tipo,
        marker_color=color,
        line_color=color,
        fillcolor=hex_a_rgba(color, 0.25),
        showlegend=False,
        boxmean=False,
        whiskerwidth=0.5,
    ), row=2, col=1)

# Panel 4 — curva de supervivencia
fig.add_trace(go.Scatter(
    x=tbf_sorted, y=survival,
    mode="lines",
    line=dict(color="#EF9F27", width=2),
    fill="tozeroy",
    fillcolor="rgba(239,159,39,0.10)",
    showlegend=False,
), row=2, col=2)

for dias, label, color in [
    (15, "15d — Alta urgencia", "#E24B4A"),
    (90, "90d — Programable",   "#378ADD"),
]:
    prob = float((df_modelo["tbf_dias"] > dias).mean())
    fig.add_shape(
        type="line", x0=dias, x1=dias,
        y0=0, y1=prob,
        line=dict(color=color, width=1.5, dash="dot"),
        row=2, col=2,
    )
    fig.add_shape(
        type="line", x0=0, x1=dias,
        y0=prob, y1=prob,
        line=dict(color=color, width=1.5, dash="dot"),
        row=2, col=2,
    )
    fig.add_annotation(
        x=dias, y=prob + 0.05,
        text=f"{label}: {prob*100:.0f}% sobreviven",
        font=dict(size=10, color=color),
        showarrow=False,
        xanchor="left",
        row=2, col=2,
    )

fig.update_layout(
    title=dict(
        text="Distribución del target — tbf_dias",
        font=dict(size=15, color="#e0ddd6", family="Arial"),
        x=0.01,
    ),
    barmode="stack",
    paper_bgcolor="#0d1117",
    plot_bgcolor="#0d1117",
    font=dict(color="#e0ddd6", family="Arial", size=11),
    height=700,
    margin=dict(t=80, b=40, l=80, r=60),
    legend=dict(
        bgcolor="#1a1a1a",
        bordercolor="#333333",
        borderwidth=1,
        font=dict(size=11),
        orientation="h",
        y=-0.05, x=0.3,
    )
)

fig.update_xaxes(gridcolor="#222222", tickfont=dict(size=10), zerolinecolor="#333333")
fig.update_yaxes(gridcolor="#222222", tickfont=dict(size=10), zerolinecolor="#333333")
fig.update_xaxes(title_text="Días",       row=1, col=1)
fig.update_xaxes(title_text="Días",       row=1, col=2)
fig.update_xaxes(title_text="Días",       row=2, col=2)
fig.update_yaxes(title_text="Frecuencia", row=1, col=1)
fig.update_yaxes(title_text="TBF (días)", row=2, col=1)
fig.update_yaxes(title_text="P(TBF > t)", row=2, col=2, tickformat=".0%")
fig.update_xaxes(range=[0, 200],          row=2, col=2)
fig.show()

# ── Resultados ─────────────────────────────────────────────────
stats    = df_modelo["tbf_dias"].describe()
skewness = df_modelo["tbf_dias"].skew()

print(f"Registros df_modelo          : {len(df_modelo)}")
print(f"Asimetría (skewness)         : {skewness:.2f} → distribución muy sesgada a la derecha")
print(f"Media                        : {stats['mean']:.1f} días")
print(f"Mediana                      : {stats['50%']:.1f} días")
print(f"Desviación estándar          : {stats['std']:.1f} días")
print(f"Máximo                       : {stats['max']:.0f} días")
print()
print(f"Probabilidades en los cortes de clasificación:")
for dias, clase in [(15, "Alta urgencia"), (90, "Programable/Preventivo")]:
    prob = (df_modelo["tbf_dias"] <= dias).mean()
    print(f"  P(TBF <= {dias:>3}d) = {prob*100:.1f}%  → {clase}")
print()
print(f"TBF mediano por tipo de equipo:")
for _, row in medianas_equipo.sort_values("mediana").iterrows():
    print(f"  {row['tipo']:<20} {row['mediana']:.0f} días")

Registros df_modelo          : 1198
Asimetría (skewness)         : 4.18 → distribución muy sesgada a la derecha
Media                        : 38.9 días
Mediana                      : 20.0 días
Desviación estándar          : 60.7 días
Máximo                       : 631 días

Probabilidades en los cortes de clasificación:
  P(TBF <=  15d) = 43.4%  → Alta urgencia
  P(TBF <=  90d) = 89.7%  → Programable/Preventivo

TBF mediano por tipo de equipo:
  RX Fijo              13 días
  RX Móvil             15 días
  TAC                  18 días
  Fluoro               22 días
  Angiografía          24 días
  Mamografía           33 días
  Arco C               49 días


14. Outliers en features numéricas. Boxplots de las 8 features numéricas del modelo para identificar valores extremos antes del entrenamiento. XGBoost es robusto frente a outliers por su naturaleza de particiones — no requiere normalización — pero valores extremos pueden sesgar las particiones en features con alta varianza como horas_parada_acumuladas y tbf_promedio_equipo.

In [57]:
# ── CELDA 14 — ANÁLISIS DE OUTLIERS EN FEATURES NUMÉRICAS ─────
import plotly.graph_objects as go
from plotly.subplots import make_subplots

FEATURES_NUM = {
    "fallos_30_dias"          : "Fallos 30 días",
    "tbf_promedio_equipo"     : "TBF promedio equipo",
    "tendencia_tbf"           : "Tendencia TBF",
    "edad_dias"               : "Edad (días)",
    "horas_parada_acumuladas" : "Horas parada acum.",
    "criticidad"              : "Criticidad",
    "tasa_fallo_equipo"       : "Tasa fallo equipo",
    "dias_desde_ultimo_fallo" : "Días último fallo",
}

def hex_a_rgba(hex_color, alpha=0.25):
    h = hex_color.lstrip("#")
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return f"rgba({r},{g},{b},{alpha})"

fig = make_subplots(
    rows=2, cols=4,
    subplot_titles=list(FEATURES_NUM.values()),
    vertical_spacing=0.18,
    horizontal_spacing=0.08,
)

COLOR_BOX = "#534AB7"

posiciones = [(1,1),(1,2),(1,3),(1,4),(2,1),(2,2),(2,3),(2,4)]

for (var, nombre), (row, col) in zip(FEATURES_NUM.items(), posiciones):
    serie = df_modelo[var].dropna()
    fig.add_trace(go.Box(
        y=serie,
        name=nombre,
        marker_color=COLOR_BOX,
        line_color=COLOR_BOX,
        fillcolor=hex_a_rgba(COLOR_BOX, 0.2),
        boxmean=True,
        showlegend=False,
        whiskerwidth=0.5,
    ), row=row, col=col)

fig.update_layout(
    title=dict(
        text="Distribución y outliers — features numéricas del modelo",
        font=dict(size=15, color="#e0ddd6", family="Arial"),
        x=0.01,
    ),
    paper_bgcolor="#0d1117",
    plot_bgcolor="#0d1117",
    font=dict(color="#e0ddd6", family="Arial", size=11),
    height=600,
    margin=dict(t=80, b=40, l=50, r=30),
)
fig.update_xaxes(showticklabels=False, gridcolor="#222222")
fig.update_yaxes(gridcolor="#222222", tickfont=dict(size=9), zerolinecolor="#333333")
fig.show()

# ── Resultados ─────────────────────────────────────────────────
print(f"{'Feature':<28} {'Media':>8} {'Mediana':>8} {'Std':>8} {'Max':>10} {'Outliers (>3σ)':>15}")
print("─" * 80)
for var, nombre in FEATURES_NUM.items():
    serie  = df_modelo[var].dropna()
    media  = serie.mean()
    std    = serie.std()
    outliers = ((serie - media).abs() > 3 * std).sum()
    print(f"  {nombre:<26} {media:>8.1f} {serie.median():>8.1f} "
          f"{std:>8.1f} {serie.max():>10.1f} {outliers:>10} ({outliers/len(serie)*100:.1f}%)")

Feature                         Media  Mediana      Std        Max  Outliers (>3σ)
────────────────────────────────────────────────────────────────────────────────
  Fallos 30 días                  1.2      1.0      1.4        8.0         15 (1.3%)
  TBF promedio equipo            41.7     26.1     45.2      631.0         26 (2.2%)
  Tendencia TBF                  -0.5     -0.3      1.2        2.2         10 (0.8%)
  Edad (días)                  1938.4   1757.5   1313.6     5447.0          0 (0.0%)
  Horas parada acum.            351.8    148.0    500.4     3627.0         12 (1.0%)
  Criticidad                      3.0      3.0      0.9        4.0          0 (0.0%)
  Tasa fallo equipo               0.0      0.0      0.0        0.1         31 (2.6%)
  Días último fallo              37.4     19.0     57.9      631.0         27 (2.3%)


15. UMAP Modelo A.
Reducción de dimensionalidad no lineal a 2D y 3D. El UMAP 2D muestra dos clusters separados en el espacio de features que corresponden a tipos de equipo distintos — RX Móvil y Arco C en el cluster izquierdo, TAC y RX Fijo en el derecho. El UMAP 3D revela si esa separación tiene una tercera dimensión que explique la mezcla de clases de urgencia dentro de cada cluster. El Silhouette Score cuantifica la separación real entre grupos en ambas dimensiones.

In [58]:
# ── CELDA 15 — UMAP MODELO A ──────────────────────────────────
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import silhouette_score

try:
    import umap
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "umap-learn", "-q"])
    import umap

VARS_UMAP_A = [
    "fallos_30_dias",
    "tbf_promedio_equipo",
    "tendencia_tbf",
    "criticidad",
    "edad_dias",
    "horas_parada_acumuladas",
    "tasa_fallo_equipo",
    "dias_desde_ultimo_fallo",
    "hubo_cambio_componente",
]

COLORES_PERIODO = {
    "Alta urgencia" : "#E24B4A",
    "Programable"   : "#378ADD",
    "Preventivo"    : "#1D9E75",
}

COLORES_EQUIPO = {
    "TAC"        : "#378ADD",
    "Fluoro"     : "#639922",
    "RX Fijo"    : "#7F77DD",
    "Mamografía" : "#D85A30",
    "RX Móvil"   : "#1D9E75",
    "Arco C"     : "#BA7517",
    "Angiografía": "#E24B4A",
}

le = LabelEncoder()
df_umap = df_modelo.copy()
for col in ["tipo_solucion", "tipo_equipo"]:
    df_umap[col + "_enc"] = le.fit_transform(df_umap[col].astype(str))

VARS_UMAP_A_FULL = VARS_UMAP_A + ["tipo_solucion_enc", "tipo_equipo_enc"]

df_umap_a = df_umap[VARS_UMAP_A_FULL + ["periodo_fallo", "tipo_corto"]].dropna().copy()

X_scaled = StandardScaler().fit_transform(df_umap_a[VARS_UMAP_A_FULL].values)

print("Calculando UMAP 2D...")
X_2d = umap.UMAP(
    n_components=2, n_neighbors=15,
    min_dist=0.1, random_state=42, verbose=False
).fit_transform(X_scaled)
print("UMAP 2D listo")

print("Calculando UMAP 3D...")
X_3d = umap.UMAP(
    n_components=3, n_neighbors=15,
    min_dist=0.1, random_state=42, verbose=False
).fit_transform(X_scaled)
print("UMAP 3D listo")

# ── Figura 1 — UMAP 2D (dos paneles) ──────────────────────────
fig1 = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Por periodo de fallo", "Por tipo de equipo"),
    horizontal_spacing=0.10,
)

for periodo, color in COLORES_PERIODO.items():
    mask = df_umap_a["periodo_fallo"] == periodo
    fig1.add_trace(go.Scatter(
        x=X_2d[mask, 0], y=X_2d[mask, 1],
        mode="markers", name=periodo,
        marker=dict(size=4, color=color, opacity=0.65, line=dict(width=0)),
        hovertemplate=f"<b>{periodo}</b><br>UMAP1: %{{x:.2f}}<br>UMAP2: %{{y:.2f}}<extra></extra>",
    ), row=1, col=1)

for tipo, color in COLORES_EQUIPO.items():
    mask = df_umap_a["tipo_corto"] == tipo
    fig1.add_trace(go.Scatter(
        x=X_2d[mask, 0], y=X_2d[mask, 1],
        mode="markers", name=tipo,
        marker=dict(size=4, color=color, opacity=0.65, line=dict(width=0)),
        hovertemplate=f"<b>{tipo}</b><br>UMAP1: %{{x:.2f}}<br>UMAP2: %{{y:.2f}}<extra></extra>",
    ), row=1, col=2)

fig1.update_layout(
    title=dict(
        text="UMAP 2D — Modelo A (periodo_fallo)",
        font=dict(size=15, color="#e0ddd6", family="Arial"),
        x=0.01,
    ),
    paper_bgcolor="#0d1117",
    plot_bgcolor="#0d1117",
    font=dict(color="#e0ddd6", family="Arial", size=11),
    height=520,
    margin=dict(t=80, b=50, l=60, r=40),
    legend=dict(bgcolor="#1a1a1a", bordercolor="#333333", borderwidth=1, font=dict(size=11)),
)
fig1.update_xaxes(title_text="UMAP 1", gridcolor="#222222", zerolinecolor="#333333")
fig1.update_yaxes(title_text="UMAP 2", gridcolor="#222222", zerolinecolor="#333333")
fig1.show()

# ── Figura 2 — UMAP 3D por periodo ────────────────────────────
fig2 = go.Figure()
for periodo, color in COLORES_PERIODO.items():
    mask = df_umap_a["periodo_fallo"] == periodo
    fig2.add_trace(go.Scatter3d(
        x=X_3d[mask, 0], y=X_3d[mask, 1], z=X_3d[mask, 2],
        mode="markers", name=periodo,
        marker=dict(size=3, color=color, opacity=0.7, line=dict(width=0)),
        hovertemplate=f"<b>{periodo}</b><br>U1: %{{x:.2f}}<br>U2: %{{y:.2f}}<br>U3: %{{z:.2f}}<extra></extra>",
    ))

fig2.update_layout(
    title=dict(
        text="UMAP 3D — Modelo A coloreado por periodo de fallo",
        font=dict(size=14, color="#e0ddd6", family="Arial"),
        x=0.01,
    ),
    scene=dict(
        xaxis=dict(title="UMAP 1", backgroundcolor="#0d1117", gridcolor="#222222",
                   titlefont=dict(color="#aaaaaa", size=11),
                   tickfont=dict(color="#888888", size=9)),
        yaxis=dict(title="UMAP 2", backgroundcolor="#0d1117", gridcolor="#222222",
                   titlefont=dict(color="#aaaaaa", size=11),
                   tickfont=dict(color="#888888", size=9)),
        zaxis=dict(title="UMAP 3", backgroundcolor="#0d1117", gridcolor="#222222",
                   titlefont=dict(color="#aaaaaa", size=11),
                   tickfont=dict(color="#888888", size=9)),
        bgcolor="#0d1117",
        camera=dict(eye=dict(x=1.5, y=1.5, z=1.2)),
    ),
    paper_bgcolor="#0d1117",
    font=dict(color="#e0ddd6", family="Arial"),
    legend=dict(bgcolor="#1a1a1a", bordercolor="#333333", borderwidth=1, font=dict(size=11)),
    height=650,
    margin=dict(t=60, b=20, l=20, r=20),
)
fig2.show()

# ── Figura 3 — UMAP 3D por tipo de equipo ─────────────────────
fig3 = go.Figure()
for tipo, color in COLORES_EQUIPO.items():
    mask = df_umap_a["tipo_corto"] == tipo
    fig3.add_trace(go.Scatter3d(
        x=X_3d[mask, 0], y=X_3d[mask, 1], z=X_3d[mask, 2],
        mode="markers", name=tipo,
        marker=dict(size=3, color=color, opacity=0.7, line=dict(width=0)),
        hovertemplate=f"<b>{tipo}</b><br>U1: %{{x:.2f}}<br>U2: %{{y:.2f}}<br>U3: %{{z:.2f}}<extra></extra>",
    ))

fig3.update_layout(
    title=dict(
        text="UMAP 3D — Modelo A coloreado por tipo de equipo",
        font=dict(size=14, color="#e0ddd6", family="Arial"),
        x=0.01,
    ),
    scene=dict(
        xaxis=dict(title="UMAP 1", backgroundcolor="#0d1117", gridcolor="#222222",
                   titlefont=dict(color="#aaaaaa", size=11),
                   tickfont=dict(color="#888888", size=9)),
        yaxis=dict(title="UMAP 2", backgroundcolor="#0d1117", gridcolor="#222222",
                   titlefont=dict(color="#aaaaaa", size=11),
                   tickfont=dict(color="#888888", size=9)),
        zaxis=dict(title="UMAP 3", backgroundcolor="#0d1117", gridcolor="#222222",
                   titlefont=dict(color="#aaaaaa", size=11),
                   tickfont=dict(color="#888888", size=9)),
        bgcolor="#0d1117",
        camera=dict(eye=dict(x=1.5, y=1.5, z=1.2)),
    ),
    paper_bgcolor="#0d1117",
    font=dict(color="#e0ddd6", family="Arial"),
    legend=dict(bgcolor="#1a1a1a", bordercolor="#333333", borderwidth=1, font=dict(size=11)),
    height=650,
    margin=dict(t=60, b=20, l=20, r=20),
)
fig3.show()

# ── Silhouette ─────────────────────────────────────────────────
labels_per = df_umap_a["periodo_fallo"].map(
    {"Alta urgencia": 0, "Programable": 1, "Preventivo": 2}
)
labels_equ = df_umap_a["tipo_corto"].map(
    {"TAC": 0, "Fluoro": 1, "RX Fijo": 2, "Mamografía": 3,
     "RX Móvil": 4, "Arco C": 5, "Angiografía": 6}
)
mask_val = labels_per.notna() & labels_equ.notna()

sil_per_2d = silhouette_score(X_2d[mask_val], labels_per[mask_val])
sil_equ_2d = silhouette_score(X_2d[mask_val], labels_equ[mask_val])
sil_per_3d = silhouette_score(X_3d[mask_val], labels_per[mask_val])
sil_equ_3d = silhouette_score(X_3d[mask_val], labels_equ[mask_val])

def nivel_sil(s):
    if s > 0.5:   return "Separación excelente"
    if s > 0.3:   return "Separación buena"
    if s > 0.1:   return "Separación moderada"
    return "Grupos muy mezclados"

# ── Resultados ─────────────────────────────────────────────────
print(f"{'Agrupación':<25} {'2D':>10} {'3D':>10}  Interpretación")
print("─" * 65)
for nombre, s2d, s3d in [
    ("Por periodo de fallo", sil_per_2d, sil_per_3d),
    ("Por tipo de equipo",   sil_equ_2d, sil_equ_3d),
]:
    print(f"  {nombre:<23} {s2d:>10.3f} {s3d:>10.3f}  {nivel_sil(s3d)}")

print(f"\nCentros por clase en espacio UMAP 2D:")
for periodo in ["Alta urgencia", "Programable", "Preventivo"]:
    mask = df_umap_a["periodo_fallo"] == periodo
    if mask.sum() > 0:
        cx = X_2d[mask, 0].mean()
        cy = X_2d[mask, 1].mean()
        print(f"  {periodo:<20} centro: ({cx:.2f}, {cy:.2f})")

Calculando UMAP 2D...


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



UMAP 2D listo
Calculando UMAP 3D...


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



UMAP 3D listo


Agrupación                        2D         3D  Interpretación
─────────────────────────────────────────────────────────────────
  Por periodo de fallo        -0.084     -0.070  Grupos muy mezclados
  Por tipo de equipo          -0.071     -0.080  Grupos muy mezclados

Centros por clase en espacio UMAP 2D:
  Alta urgencia        centro: (6.67, 2.61)
  Programable          centro: (6.69, 2.21)
  Preventivo           centro: (6.02, 2.15)


16. Agrupación de componentes para el modelo B. Se construye componente_agrupado a partir de la columna componente usando cuatro capas de resolución en orden: match exacto en el mapa construido desde los PDFs oficiales de los 7 equipos, fuzzy matching con umbral de 85% para capturar variantes de escritura, keywords por sistema como última instancia antes del fallback, y "falla desconocida" para registros sin componente identificado. Los componentes no resueltos por ninguna capa se registran en COMPONENTES_SIN_MAPEAR para revisión futura. La coincidencia del 84.5% con sistema_general_target confirma señal real sin data leakage.

In [59]:
# ── CELDA 16 — AGRUPACIÓN DE COMPONENTE PARA MODELO B ─────────

try:
    from rapidfuzz import process as rf_process, fuzz
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "rapidfuzz", "-q"])
    from rapidfuzz import process as rf_process, fuzz

MAPA_COMPONENTE = {

    # ── Generación y detección de Rayos X ─────────────────────
    "tubo de rayos x"                          : "generación y detección de rayos x",
    "tubo rx"                                  : "generación y detección de rayos x",
    "generador de alta tensión"                : "generación y detección de rayos x",
    "cables de alta tensión"                   : "generación y detección de rayos x",
    "colimador"                                : "generación y detección de rayos x",
    "luz de campo"                             : "generación y detección de rayos x",
    "inversor de alta frecuencia"              : "generación y detección de rayos x",
    "detector"                                 : "generación y detección de rayos x",
    "detector plano"                           : "generación y detección de rayos x",
    "detector fijo"                            : "generación y detección de rayos x",
    "detector fijo (bucky mural)"              : "generación y detección de rayos x",
    "mini detector"                            : "generación y detección de rayos x",
    "carcasa del detector"                     : "generación y detección de rayos x",
    "carcasa detector"                         : "generación y detección de rayos x",
    "rejilla anti dispersión"                  : "generación y detección de rayos x",
    "rejilla antidispersión"                   : "generación y detección de rayos x",
    "medidor de dosis"                         : "generación y detección de rayos x",
    "medidor de dosis (dap)"                   : "generación y detección de rayos x",
    "cámara de ionización"                     : "generación y detección de rayos x",
    "cámara dosimétrica"                       : "generación y detección de rayos x",
    "cámara dosimétrica (dap)"                 : "generación y detección de rayos x",
    "módulo de registro y reporte de dosis"    : "generación y detección de rayos x",
    "intensificador de imagen"                 : "generación y detección de rayos x",
    "cámara de video"                          : "generación y detección de rayos x",
    "bomba de refrigeración"                   : "generación y detección de rayos x",
    "ventilador"                               : "generación y detección de rayos x",
    "haz de radiación"                         : "generación y detección de rayos x",
    "haz radiación"                            : "generación y detección de rayos x",
    "laser guía"                               : "generación y detección de rayos x",
    "inversor"                                 : "generación y detección de rayos x",
    "tanque de alta tensión"                   : "generación y detección de rayos x",
    "transformador inversor"                   : "generación y detección de rayos x",
    "sensores del tubo de rayos x"             : "generación y detección de rayos x",
    "unidad de refrigeración del tubo"         : "generación y detección de rayos x",
    "sensores de temperatura"                  : "generación y detección de rayos x",
    "panel de detector"                        : "generación y detección de rayos x",
    "tarjetas de adquisición"                  : "generación y detección de rayos x",
    "tarjetas de adquisición (adc)"            : "generación y detección de rayos x",
    "módulo de detector"                       : "generación y detección de rayos x",
    "mylar"                                    : "generación y detección de rayos x",
    "tarjetas de reconstrucción"               : "generación y detección de rayos x",
    "holder"                                   : "generación y detección de rayos x",
    "pines de carga del detector"              : "generación y detección de rayos x",
    "pines de carfa del detector"              : "generación y detección de rayos x",
    "pines carga detector"                     : "generación y detección de rayos x",
    "baterías detector"                        : "generación y detección de rayos x",
    "bateria detector"                         : "generación y detección de rayos x",
    "baterías de control remoto"               : "generación y detección de rayos x",
    "tierra detector mesa"                     : "generación y detección de rayos x",
    "tierra detector"                          : "generación y detección de rayos x",
    "perillas del colimador"                   : "generación y detección de rayos x",
    "vidrio colimador"                         : "generación y detección de rayos x",
    "banco de capacitores"                     : "generación y detección de rayos x",
    "fuente detector mesa"                     : "generación y detección de rayos x",
    "fuente detector bucky mural"              : "generación y detección de rayos x",
    "dap"                                      : "generación y detección de rayos x",
    "modulos de detección"                     : "generación y detección de rayos x",
    "módulos de detección"                     : "generación y detección de rayos x",
    "medio de enfriamiento"                    : "generación y detección de rayos x",

    # ── Mecánico y de posicionamiento ─────────────────────────
    "brazo articulado"                         : "mecánico y de posicionamiento",
    "brazo"                                    : "mecánico y de posicionamiento",
    "brazo giratorio"                          : "mecánico y de posicionamiento",
    "brazo de soporte horizontal"              : "mecánico y de posicionamiento",
    "riel"                                     : "mecánico y de posicionamiento",
    "riel del techo"                           : "mecánico y de posicionamiento",
    "rieles del arco"                          : "mecánico y de posicionamiento",
    "cabezal porta tubo"                       : "mecánico y de posicionamiento",
    "ruedas"                                   : "mecánico y de posicionamiento",
    "ruedas motorizadas"                       : "mecánico y de posicionamiento",
    "ruedas mecánicas"                         : "mecánico y de posicionamiento",
    "ruedas mecanicas"                         : "mecánico y de posicionamiento",
    "manija de empuje"                         : "mecánico y de posicionamiento",
    "manija del brazo"                         : "mecánico y de posicionamiento",
    "manija de desplazamiento"                 : "mecánico y de posicionamiento",
    "manija de movimiento"                     : "mecánico y de posicionamiento",
    "perilla de movimiento steering"           : "mecánico y de posicionamiento",
    "mecanismo de parqueo (hook)"              : "mecánico y de posicionamiento",
    "mecanismo de perilla de movimiento steering" : "mecánico y de posicionamiento",
    "mecanismo de freno"                       : "mecánico y de posicionamiento",
    "freno central de bloqueo"                 : "mecánico y de posicionamiento",
    "frenos mecánicos"                         : "mecánico y de posicionamiento",
    "frenos electromagneticos"                 : "mecánico y de posicionamiento",
    "freno electromagnético"                   : "mecánico y de posicionamiento",
    "liberación de emergencia del sistema de frenos" : "mecánico y de posicionamiento",
    "potenciómetros"                           : "mecánico y de posicionamiento",
    "potenciometros"                           : "mecánico y de posicionamiento",
    "encoders"                                 : "mecánico y de posicionamiento",
    "enconders"                                : "mecánico y de posicionamiento",
    "encoder"                                  : "mecánico y de posicionamiento",
    "encoders de posición"                     : "mecánico y de posicionamiento",
    "sensores de límite"                       : "mecánico y de posicionamiento",
    "sensores piezoeléctricos"                 : "mecánico y de posicionamiento",
    "sensores de colisión"                     : "mecánico y de posicionamiento",
    "sensores de posición"                     : "mecánico y de posicionamiento",
    "motores de movimiento"                    : "mecánico y de posicionamiento",
    "motores de la mesa"                       : "mecánico y de posicionamiento",
    "motores ruedas"                           : "mecánico y de posicionamiento",
    "controlador de motor"                     : "mecánico y de posicionamiento",
    "table top"                                : "mecánico y de posicionamiento",
    "de table top"                             : "mecánico y de posicionamiento",
    "bucky mural"                              : "mecánico y de posicionamiento",
    "bucky de mesa"                            : "mecánico y de posicionamiento",
    "manija del bucky"                         : "mecánico y de posicionamiento",
    "detent"                                   : "mecánico y de posicionamiento",
    "rodamientos"                              : "mecánico y de posicionamiento",
    "rodamientos de la c"                      : "mecánico y de posicionamiento",
    "guías y rodamientos"                      : "mecánico y de posicionamiento",
    "estructura metálica"                      : "mecánico y de posicionamiento",
    "estructura metalica"                      : "mecánico y de posicionamiento",
    "topes mecánicos de movimiento"            : "mecánico y de posicionamiento",
    "topes límites"                            : "mecánico y de posicionamiento",
    "carro longitudinal con soporte de rx"     : "mecánico y de posicionamiento",
    "base de la unidad con accionamiento de elevación y basculación" : "mecánico y de posicionamiento",
    "arco en c"                                : "mecánico y de posicionamiento",
    "columna elevadora"                        : "mecánico y de posicionamiento",
    "columna de monitores"                     : "mecánico y de posicionamiento",
    "columna de soporte del generador"         : "mecánico y de posicionamiento",
    "columna"                                  : "mecánico y de posicionamiento",
    "soporte del tubo"                         : "mecánico y de posicionamiento",
    "soporte del detector"                     : "mecánico y de posicionamiento",
    "soporte de rx"                            : "mecánico y de posicionamiento",
    "sistema de soporte de las pantallas"      : "mecánico y de posicionamiento",
    "unidad de compresión"                     : "mecánico y de posicionamiento",
    "reposacabezas"                            : "mecánico y de posicionamiento",
    "mesa de apoyo del detector"               : "mecánico y de posicionamiento",
    "soporte vertical"                         : "mecánico y de posicionamiento",
    "cadena deslizante"                        : "mecánico y de posicionamiento",
    "soporte de angulación"                    : "mecánico y de posicionamiento",
    "rac controller"                           : "mecánico y de posicionamiento",
    "sistema de inclinación del gantry"        : "mecánico y de posicionamiento",
    "láser de posicionamiento"                 : "mecánico y de posicionamiento",
    "módulo de movimiento horizontal"          : "mecánico y de posicionamiento",
    "módulo de movimiento vertical"            : "mecánico y de posicionamiento",
    "encoder / potenciometro de la mesa"       : "mecánico y de posicionamiento",
    "potenciometros arco en c"                 : "mecánico y de posicionamiento",
    "módulo de potencia del motor del stand"   : "mecánico y de posicionamiento",
    "cinta metálica del detector"              : "mecánico y de posicionamiento",
    "cinta metalica"                           : "mecánico y de posicionamiento",
    "seguro de soporte"                        : "mecánico y de posicionamiento",
    "puerto hand switch"                       : "mecánico y de posicionamiento",

    # ── Control e interfaz de usuario ─────────────────────────
    "hand switch"                              : "control e interfaz de usuario",
    "connector hand switch"                    : "control e interfaz de usuario",
    "conexion  hand switch"                    : "control e interfaz de usuario",
    "conexión hand switch"                     : "control e interfaz de usuario",
    "control remoto"                           : "control e interfaz de usuario",
    "interruptor de pedal"                     : "control e interfaz de usuario",
    "interruptor de puntapié"                  : "control e interfaz de usuario",
    "interruptor de pedal mesa"                : "control e interfaz de usuario",
    "interruptor de pedal fluoroscopia"        : "control e interfaz de usuario",
    "interruptor manual"                       : "control e interfaz de usuario",
    "pedales"                                  : "control e interfaz de usuario",
    "pedales de irradiación mesa"              : "control e interfaz de usuario",
    "pedales de irradiación sala"              : "control e interfaz de usuario",
    "pedales de compresión y movimiento"       : "control e interfaz de usuario",
    "pedal de irradiación"                     : "control e interfaz de usuario",
    "pulsadores de micromovimientos"           : "control e interfaz de usuario",
    "manijas con pulsadores"                   : "control e interfaz de usuario",
    "botones de frenos"                        : "control e interfaz de usuario",
    "teclado de frenos (arco)"                 : "control e interfaz de usuario",
    "panel de mando (porta monitor)"           : "control e interfaz de usuario",
    "panel de mando (mamounit)"                : "control e interfaz de usuario",
    "paneles de control"                       : "control e interfaz de usuario",
    "panel de operaciones del gantry"          : "control e interfaz de usuario",
    "panel de control"                         : "control e interfaz de usuario",
    "unidad de control (arco en c)"            : "control e interfaz de usuario",
    "unidad de control mesa"                   : "control e interfaz de usuario",
    "consola de movimiento / radiación mesa"   : "control e interfaz de usuario",
    "consola de movimiento / radiación sala"   : "control e interfaz de usuario",
    "consola de exploración"                   : "control e interfaz de usuario",
    "consola"                                  : "control e interfaz de usuario",
    "modulo de encendido"                      : "control e interfaz de usuario",
    "teclado"                                  : "control e interfaz de usuario",
    "mouse"                                    : "control e interfaz de usuario",
    "ratón"                                    : "control e interfaz de usuario",
    "raton"                                    : "control e interfaz de usuario",
    "monitor"                                  : "control e interfaz de usuario",
    "monitores"                                : "control e interfaz de usuario",
    "monitor live mesa"                        : "control e interfaz de usuario",
    "monitor live sala"                        : "control e interfaz de usuario",
    "monitor ref mesa"                         : "control e interfaz de usuario",
    "monitor ref sala"                         : "control e interfaz de usuario",
    "monitor de visualización"                 : "control e interfaz de usuario",
    "pantalla de soporte"                      : "control e interfaz de usuario",
    "pantalla táctil"                          : "control e interfaz de usuario",
    "pantalla tactil"                          : "control e interfaz de usuario",
    "pantalla vivo (en la sala de procedimiento)" : "control e interfaz de usuario",
    "botón on / off"                           : "control e interfaz de usuario",
    "botón de exposición"                      : "control e interfaz de usuario",
    "botón de descomprimir"                    : "control e interfaz de usuario",
    "boton de parada de emergencia"            : "control e interfaz de usuario",
    "botón parada de emergencia"               : "control e interfaz de usuario",
    "entradas usb"                             : "control e interfaz de usuario",
    "cable vga"                                : "control e interfaz de usuario",
    "lámparas de indicación / exposición"      : "control e interfaz de usuario",
    "proyector"                                : "control e interfaz de usuario",
    "guía del paciente"                        : "control e interfaz de usuario",
    "joystick del arco"                        : "control e interfaz de usuario",
    "joystick"                                 : "control e interfaz de usuario",

    # ── Procesamiento y almacenamiento ────────────────────────
    "cpu"                                      : "procesamiento y almacenamiento",
    "gpu"                                      : "procesamiento y almacenamiento",
    "disco duro"                               : "procesamiento y almacenamiento",
    "ram"                                      : "procesamiento y almacenamiento",
    "tarjeta madre"                            : "procesamiento y almacenamiento",
    "tarjeta de video"                         : "procesamiento y almacenamiento",
    "software del equipo"                      : "procesamiento y almacenamiento",
    "software propio del equipo"               : "procesamiento y almacenamiento",
    "protocolos del equipo"                    : "procesamiento y almacenamiento",
    "protocolos"                               : "procesamiento y almacenamiento",
    "aplicaciones clínicas"                    : "procesamiento y almacenamiento",
    "aplicación 3d"                            : "procesamiento y almacenamiento",
    "interfaz de software"                     : "procesamiento y almacenamiento",
    "convertidores de video"                   : "procesamiento y almacenamiento",
    "convertidor de video"                     : "procesamiento y almacenamiento",
    "convertidor de señal de video"            : "procesamiento y almacenamiento",
    "módulo en la cpu (tarjeta sd)"            : "procesamiento y almacenamiento",
    "distribuidor de señal"                    : "procesamiento y almacenamiento",
    "pantallas de referencia"                  : "procesamiento y almacenamiento",
    "cable de video"                           : "procesamiento y almacenamiento",
    "adaptador de conexión de video"           : "procesamiento y almacenamiento",
    "adaptador de conexión de video (dvi-hdmi)": "procesamiento y almacenamiento",
    "puerto de video"                          : "procesamiento y almacenamiento",
    "cable de transmisión de imagen"           : "procesamiento y almacenamiento",
    "tarjetas de procesamiento"                : "procesamiento y almacenamiento",
    "módulo de scan"                           : "procesamiento y almacenamiento",
    "computadora de almacenamiento"            : "procesamiento y almacenamiento",
    "módulo de procesamiento"                  : "procesamiento y almacenamiento",
    "módulo de comunicación de imagen"         : "procesamiento y almacenamiento",
    "borrado de archivos"                      : "procesamiento y almacenamiento",
    "pantalla vivo sala"                       : "procesamiento y almacenamiento",
    "pantalla vivo"                            : "procesamiento y almacenamiento",
    "módulo de reconstrucción"                 : "procesamiento y almacenamiento",
    "modulo de reconstrucción"                 : "procesamiento y almacenamiento",

    # ── Comunicaciones ────────────────────────────────────────
    "plataforma clínica (pacs)"                : "comunicaciones",
    "plataforma clinica (pacs)"                : "comunicaciones",
    "plataforma clínica"                       : "comunicaciones",
    "plataforma clinica"                       : "comunicaciones",
    "cable de red"                             : "comunicaciones",
    "nodo de red"                              : "comunicaciones",
    "módulo wifi"                              : "comunicaciones",
    "módulo wifi detector"                     : "comunicaciones",
    "módulo wifi clínica"                      : "comunicaciones",
    "sensor infrarrojo"                        : "comunicaciones",
    "cable de intercomunicación"               : "comunicaciones",
    "conector cable de intercomunicación"      : "comunicaciones",
    "conexión wlan y lan"                      : "comunicaciones",
    "conexión de red"                          : "comunicaciones",
    "conexión del equipo a la red"             : "comunicaciones",
    "conexión consola - equipo"                : "comunicaciones",
    "network switch"                           : "comunicaciones",
    "puerto de red"                            : "comunicaciones",
    "micrófono"                                : "comunicaciones",
    "altavoces"                                : "comunicaciones",
    "sistema de intercomunicación"             : "comunicaciones",
    "módulo de comunicación consola - equipo"  : "comunicaciones",
    "cable de conexión cuarto de equipos - consola (cable dvi)" : "comunicaciones",
    "tarjeta de red"                           : "comunicaciones",
    "conexión mu - ws"                         : "comunicaciones",
    "conexion mu - ws"                         : "comunicaciones",

    # ── Eléctrico del equipo ──────────────────────────────────
    "interruptor general"                      : "eléctrico del equipo",
    "fusibles"                                 : "eléctrico del equipo",
    "fusible"                                  : "eléctrico del equipo",
    "ups"                                      : "eléctrico del equipo",
    "baterías internas"                        : "eléctrico del equipo",
    "cable de poder retráctil"                 : "eléctrico del equipo",
    "cable de poder retractil"                 : "eléctrico del equipo",
    "clavija"                                  : "eléctrico del equipo",
    "baterías cpu"                             : "eléctrico del equipo",
    "conductor de protección"                  : "eléctrico del equipo",
    "uniones equipotenciales del chasis y mesa": "eléctrico del equipo",
    "uniones equipotenciales del chasis"       : "eléctrico del equipo",
    "panel de distribución interno"            : "eléctrico del equipo",
    "panel de distribución"                    : "eléctrico del equipo",
    "gabinetes del equipo"                     : "eléctrico del equipo",
    "suministro eléctrico"                     : "eléctrico del equipo",
    "tarjeta mcu"                              : "eléctrico del equipo",
    "tarjeta can i/o"                          : "eléctrico del equipo",
    "tarjeta de control del arco (ge)"         : "eléctrico del equipo",
    "tarjeta de control interruptor manual y de pedal" : "eléctrico del equipo",
    "electrónica de alimentación"              : "eléctrico del equipo",
    "cable de intercomunicación (tc)"          : "eléctrico del equipo",
    "botón de encendido y apagado"             : "eléctrico del equipo",
    "transformador"                            : "eléctrico del equipo",
    "breakers del equipo"                      : "eléctrico del equipo",
    "escobillas de potencia"                   : "eléctrico del equipo",
    "unidad de alimentación de alta tensión"   : "eléctrico del equipo",
    "unidad de alimentación de baja tensión"   : "eléctrico del equipo",
    "fuente de alimentacion externa"           : "eléctrico del equipo",
    "receptáculo"                              : "eléctrico del equipo",
    "interruptores de límite"                  : "eléctrico del equipo",
    "fuente 24v"                               : "eléctrico del equipo",
    "fuente 12v"                               : "eléctrico del equipo",
    "power board"                              : "eléctrico del equipo",
    "tarjeta de distribución (d883)"           : "eléctrico del equipo",
    "d840"                                     : "eléctrico del equipo",
    "d50"                                      : "eléctrico del equipo",
    "boton encendido / apagado"                : "eléctrico del equipo",
    "extractores gabinetes"                    : "eléctrico del equipo",
    "ventiladores gabinetes"                   : "eléctrico del equipo",
    "chiller"                                  : "eléctrico del equipo",
    "módulo de alimentación (stnavi box)"      : "eléctrico del equipo",
    "interruptor on / off (stnavi box)"        : "eléctrico del equipo",
    "módulo psu"                               : "eléctrico del equipo",
    "tarjeta controladora principal de movimiento (cbs y cba)" : "eléctrico del equipo",
    "tarjetas controladoras de motores"        : "eléctrico del equipo",
    "cableado y conexiones asociadas"          : "eléctrico del equipo",
    "tarjeta de control del gantry (gms y gscb)" : "eléctrico del equipo",
    "tarjeta de imagen (sct2 y tgpu)"          : "eléctrico del equipo",
    "unidad de control"                        : "eléctrico del equipo",
    "tablero electronico"                      : "eléctrico del equipo",
    "tablero electrónico"                      : "eléctrico del equipo",
    "cableados del equipo"                     : "eléctrico del equipo",

    # ── Seguridad del paciente ────────────────────────────────
    "carcasa del equipo"                       : "seguridad del paciente",
    "biombo"                                   : "seguridad del paciente",
    "goma de descarga de voltaje"              : "seguridad del paciente",
    "indicador de radiación"                   : "seguridad del paciente",
    "alarmas del equipo"                       : "seguridad del paciente",
    "protector facial"                         : "seguridad del paciente",
    "sistema de liberación de emergencia"      : "seguridad del paciente",
    "palanca de liberación"                    : "seguridad del paciente",
    "interbloqueos por movimiento"             : "seguridad del paciente",
    "bloqueo de movimiento"                    : "seguridad del paciente",
    "correas de sujeción"                      : "seguridad del paciente",
    "parada de emergencia"                     : "seguridad del paciente",

    # ── Accesorios ────────────────────────────────────────────
    "reposapies ajustable"                     : "accesorios",
    "asiento bebe"                             : "accesorios",
    "soporte de chalecos"                      : "accesorios",
    "protector de pantalla"                    : "accesorios",
    "robot de biopsia"                         : "accesorios",
    "portaagujas"                              : "accesorios",
    "pistola de biopsia"                       : "accesorios",
    "kit de calibración"                       : "accesorios",
    "mampara antirradiación"                   : "accesorios",
    "unidad de esterotaxia"                    : "accesorios",
    "bandeja de amplificación o magnificación" : "accesorios",
    "bandeja de compresión"                    : "accesorios",
    "colchoneta camilla"                       : "accesorios",
    "soporte del brazo"                        : "accesorios",
    "fladón plomado"                           : "accesorios",
    "monitor de signos vitales"                : "accesorios",
    "inyector de contraste"                    : "accesorios",
    "phantom"                                  : "accesorios",
    "cable ecg"                                : "accesorios",
    "conexión monitor ecg - equipo"            : "accesorios",
    "monitor flurotac"                         : "accesorios",
    "aguja de calibración"                     : "accesorios",
    "laser externo"                            : "accesorios",
    "e-box"                                    : "accesorios",
    "velcros"                                  : "accesorios",
    "sujetadores"                              : "accesorios",
    "conexión accesorio"                       : "accesorios",
    "conexion accesorio"                       : "accesorios",

    # ── Usuario ───────────────────────────────────────────────
    "uso incorrecto del equipo"                : "usuario",
    "mal uso del equipo"                       : "usuario",
    "error en configuración de protocolos"     : "usuario",
    "borrado de archivos"                      : "usuario",
    "correcion por mal uso"                    : "usuario",
    "corrección por mal uso"                   : "usuario",
    "asesoría"                                 : "usuario",
    "asesoria"                                 : "usuario",

    # ── Falla desconocida ─────────────────────────────────────
    "falla desconocida"                        : "falla desconocida",

    # ── Otro ──────────────────────────────────────────────────
    "configuracion ip"                         : "otro",
    "configuración ip"                         : "otro",
    "error en accionamiento"                   : "otro",
    "apagón componente"                        : "otro",
    "mensaje de error"                         : "otro",
    "scoutview"                                : "otro",
    "articio en la imagen"                     : "otro",
    "artificio en la imagen"                   : "otro",
    "nivel de tolerancia"                      : "otro",
    "sensor de alfombrilla"                    : "otro",
    "aire acondicionado"                       : "otro",
}

KEYWORDS_SISTEMA = {
    "generación y detección de rayos x": [
        "tubo de rayos", "tubo rx", "generador de alta tensión", "detector",
        "colimador", "rejilla", "dosis", "dosimétric", "ionización",
        "intensificador", "mylar", "radiación", "haz", "inversor", "laser guía",
        "refrigeración tubo", "adquisición",
    ],
    "mecánico y de posicionamiento": [
        "brazo", "rueda", "manija", "freno", "encoder", "potenciómetr",
        "potenciometr", "riel", "bucky", "gantry", "columna", "soporte",
        "mesa", "rodamiento", "motor", "tope", "estructura", "arco en c",
        "cabezal", "elevador", "desplazamiento", "steering",
    ],
    "control e interfaz de usuario": [
        "hand switch", "pedal", "consola", "panel de mando", "panel de control",
        "paneles de control", "teclado", "mouse", "ratón", "monitor",
        "pantalla táctil", "pantalla tactil", "control remoto", "pulsador",
        "botón", "boton", "interruptor de pedal", "interruptor manual",
        "entradas usb", "proyector", "lámpara", "guía del paciente",
    ],
    "procesamiento y almacenamiento": [
        "cpu", "gpu", "disco duro", "ram", "software", "protocolo",
        "tarjeta madre", "tarjeta de video", "procesamiento", "almacenamiento",
        "convertidor de video", "convertidor de señal", "cable de video",
        "adaptador de video", "módulo de imagen", "aplicación",
    ],
    "comunicaciones": [
        "pacs", "red", "wifi", "cable de red", "nodo", "intercomunicación",
        "wlan", "lan", "micrófono", "altavoz", "switch", "conexión de red",
        "tarjeta de red",
    ],
    "eléctrico del equipo": [
        "fusible", "ups", "batería", "clavija", "cable de poder",
        "interruptor general", "transformador", "breaker", "gabinete",
        "alimentación", "distribución", "electrónica", "tarjeta de control",
        "equipotencial", "conductor de protección", "power board",
        "encendido", "suministro eléctrico",
    ],
    "seguridad del paciente": [
        "parada de emergencia", "carcasa", "biombo", "protector",
        "alarma", "interbloqueo", "bloqueo", "correa", "palanca de liberación",
        "liberación de emergencia",
    ],
    "accesorios": [
        "biopsia", "phantom", "ecg", "signos vitales", "inyector",
        "mampara", "portaagujas", "bandeja", "chaleco", "soporte del brazo",
        "colchoneta", "fladón",
    ],
    "usuario": [
        "mal uso", "uso incorrecto", "asesor", "capacitación", "borrado",
    ],
}

UMBRAL_FUZZY           = 85
CLAVES_MAPA            = list(MAPA_COMPONENTE.keys())
COMPONENTES_SIN_MAPEAR = []

def agrupar_componente(comp_raw):
    if not isinstance(comp_raw, str) or comp_raw.strip() == "" or comp_raw.strip() == "nan":
        return "falla desconocida"
    comp = comp_raw.strip().lower()
    if comp in MAPA_COMPONENTE:
        return MAPA_COMPONENTE[comp]
    resultado = rf_process.extractOne(
        comp, CLAVES_MAPA,
        scorer=fuzz.token_sort_ratio,
        score_cutoff=UMBRAL_FUZZY,
    )
    if resultado is not None:
        mejor_clave, score, _ = resultado
        return MAPA_COMPONENTE[mejor_clave]
    for sistema, keywords in KEYWORDS_SISTEMA.items():
        for kw in keywords:
            if kw in comp:
                return sistema
    COMPONENTES_SIN_MAPEAR.append(comp)
    return "otro"

df_modelo["componente_agrupado"] = (
    df_modelo["componente"]
    .astype(str)
    .str.strip()
    .str.lower()
    .apply(agrupar_componente)
)

# ── Resultados ─────────────────────────────────────────────────
print(f"Distribución componente_agrupado ({df_modelo['componente_agrupado'].nunique()} grupos):")
print(df_modelo["componente_agrupado"].value_counts().to_string())

otros = df_modelo[df_modelo["componente_agrupado"] == "otro"]["componente"] \
    .astype(str).str.strip().str.lower().value_counts()
print(f"\nComponentes en 'otro': {len(otros)}")
if len(otros) > 0:
    print(otros.to_string())

if COMPONENTES_SIN_MAPEAR:
    from collections import Counter
    print(f"\n⚠ Componentes nuevos sin mapear:")
    for comp, n in Counter(COMPONENTES_SIN_MAPEAR).most_common():
        print(f"  '{comp}'  ×{n}")
else:
    print("\n✓ Todos los componentes resueltos por exacto, fuzzy o keywords")

coincidencia = (df_modelo["componente_agrupado"] == df_modelo["sistema_general_target"]).mean()
print(f"\nCoincidencia vs sistema_general_target: {coincidencia:.1%}")
print("→ Coincidencia parcial esperada — no hay data leakage")

Distribución componente_agrupado (11 grupos):
componente_agrupado
generación y detección de rayos x    217
mecánico y de posicionamiento        191
procesamiento y almacenamiento       189
control e interfaz de usuario        152
eléctrico del equipo                 121
comunicaciones                       108
falla desconocida                     96
accesorios                            47
seguridad del paciente                40
usuario                               22
otro                                  15

Componentes en 'otro': 8
componente
configuracion ip          5
error en accionamiento    2
mensaje de error          2
apagón componente         2
scoutview                 1
sensor de alfombrilla     1
articio en la imagen      1
configuración ip          1

✓ Todos los componentes resueltos por exacto, fuzzy o keywords

Coincidencia vs sistema_general_target: 84.5%
→ Coincidencia parcial esperada — no hay data leakage


17. Variables de entrada ModeloB. Tabla de las 10 features del Modelo B clasificadas por tipo e importancia esperada. componente_agrupado es la feature más importante — identifica el sistema físico del componente que falló. Las variables categóricas tipo_solucion y tipo_equipo complementan la señal de componente_agrupado con el contexto de la intervención.

In [60]:
# ── CELDA 17 — VARIABLES DE ENTRADA MODELO B ──────────────────
import plotly.graph_objects as go

variables_b = pd.DataFrame([
    ["tipo_solucion",           "Categórica", "Tipo de solución aplicada al mantenimiento",       "Alta"],
    ["hubo_cambio_componente",  "Binaria",    "1 si se reemplazó algún componente, 0 si no",      "Alta"],
    ["tipo_equipo",             "Categórica", "Tipo de equipo médico",                            "Alta"],
    ["criticidad",              "Numérica",   "Índice compuesto de gravedad (1-5)",               "Media"],
    ["fallos_30_dias",          "Numérica",   "Fallos del equipo en los últimos 30 días",         "Media"],
    ["edad_dias",               "Numérica",   "Días instalado al momento del fallo",              "Media"],
    ["tbf_promedio_equipo",     "Numérica",   "TBF promedio histórico del equipo",                "Media"],
    ["tendencia_tbf",           "Numérica",   "Desviación del TBF actual respecto al histórico",  "Media"],
    ["horas_parada_acumuladas", "Numérica",   "Total horas fuera de servicio acumuladas",         "Media"],
    ["componente_agrupado",     "Categórica", "Sistema al que pertenece el componente que falló", "Muy alta"],
], columns=["Variable", "Tipo", "Descripción", "Importancia esperada"])

color_tipo = {"Numérica": "#1a3a5c", "Categórica": "#1a3a2a", "Binaria": "#3a2a1a"}
color_imp  = {"Muy alta": "#1a5a2a", "Alta": "#1a4a2a", "Media": "#3a3a1a"}

fig = go.Figure(data=[go.Table(
    columnwidth=[220, 120, 380, 180],
    header=dict(
        values=["<b>Variable</b>", "<b>Tipo</b>", "<b>Descripción</b>", "<b>Importancia esperada</b>"],
        fill_color="#1a4a2a",
        font=dict(color="white", size=13, family="Arial"),
        align="left",
        height=36,
    ),
    cells=dict(
        values=[
            variables_b["Variable"].tolist(),
            variables_b["Tipo"].tolist(),
            variables_b["Descripción"].tolist(),
            variables_b["Importancia esperada"].tolist(),
        ],
        fill_color=[
            ["#0d1117"] * len(variables_b),
            [color_tipo.get(t, "#1a1a1a") for t in variables_b["Tipo"]],
            ["#0d1117"] * len(variables_b),
            [color_imp.get(i,  "#1a1a1a") for i in variables_b["Importancia esperada"]],
        ],
        font=dict(color=["#e0ddd6", "white", "#cccccc", "white"], size=12, family="Arial"),
        align="left",
        height=38,
    )
)])

fig.update_layout(
    title=dict(
        text="Variables de entrada — Modelo B (sistema_general_target) — 10 clases",
        font=dict(size=14, color="#e0ddd6", family="Arial"),
        x=0.01,
    ),
    margin=dict(t=50, b=10, l=0, r=0),
    paper_bgcolor="#0d1117",
    height=460,
)
fig.show()

# ── Resultados ─────────────────────────────────────────────────
print(f"Total variables Modelo B : {len(variables_b)}")
print(f"Numéricas                : {(variables_b['Tipo'] == 'Numérica').sum()}")
print(f"Categóricas              : {(variables_b['Tipo'] == 'Categórica').sum()}")
print(f"Binarias                 : {(variables_b['Tipo'] == 'Binaria').sum()}")
print(f"Target                   : {TARGET_SISTEMA} — 10 clases")

Total variables Modelo B : 10
Numéricas                : 6
Categóricas              : 3
Binarias                 : 1
Target                   : sistema_general_target — 10 clases


18. Distribución del target sistema_general. Distribución global de las 10 clases del Modelo B y su proporción por tipo de equipo. Las clases con menos del 5% de registros están subrepresentadas y justifican el uso de **compute_sample_weight**. El panel de proporciones revela qué sistemas son más frecuentes por tipo de equipo, lo que permite validar que la distribución es coherente con el conocimiento técnico de los equipos.

In [61]:
# ── CELDA 18 — DISTRIBUCIÓN TARGET SISTEMA_GENERAL_TARGET ─────
import plotly.graph_objects as go
from plotly.subplots import make_subplots

if "tipo_corto" not in df_modelo.columns:
    df_modelo["tipo_corto"] = df_modelo["tipo_equipo"].map(NOMBRES_CORTOS)

COLORES_SISTEMA = {
    "mecánico y de posicionamiento"      : "#378ADD",
    "generación y detección de rayos x"  : "#EF9F27",
    "eléctrico del equipo"               : "#E24B4A",
    "control e interfaz de usuario"      : "#1D9E75",
    "procesamiento y almacenamiento"     : "#7F77DD",
    "comunicaciones"                     : "#D85A30",
    "falla desconocida"                  : "#888888",
    "accesorios"                         : "#639922",
    "seguridad del paciente"             : "#534AB7",
    "usuario"                            : "#CC6688",
}

orden   = list(df_modelo[TARGET_SISTEMA].value_counts().index)
colores = [COLORES_SISTEMA.get(s, "#888888") for s in orden]
total   = len(df_modelo[TARGET_SISTEMA].dropna())
valores = [df_modelo[TARGET_SISTEMA].value_counts()[s] for s in orden]

tipos_orden = ["TAC", "Angiografía", "Arco C", "RX Fijo", "RX Móvil", "Fluoro", "Mamografía"]

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        "Distribución global",
        "Proporción por tipo de equipo",
    ),
    horizontal_spacing=0.12,
)

fig.add_trace(go.Bar(
    x=[s.replace(" y ", "<br>y ") for s in orden],
    y=valores,
    marker_color=colores,
    marker_line_width=0,
    text=[str(v) for v in valores],
    textposition="outside",
    textfont=dict(size=11, color="#aaaaaa"),
    showlegend=False,
), row=1, col=1)

for sistema, color in COLORES_SISTEMA.items():
    if sistema not in df_modelo[TARGET_SISTEMA].values:
        continue
    vals = []
    for tipo in tipos_orden:
        grupo      = df_modelo[df_modelo["tipo_corto"] == tipo]
        total_tipo = len(grupo[grupo[TARGET_SISTEMA].notna()])
        n          = (grupo[TARGET_SISTEMA] == sistema).sum()
        vals.append(round(n / total_tipo * 100, 1) if total_tipo > 0 else 0)
    fig.add_trace(go.Bar(
        x=tipos_orden, y=vals,
        name=sistema,
        marker_color=color,
        showlegend=True,
    ), row=1, col=2)

fig.update_layout(
    title=dict(
        text="Distribución del target — sistema_general_target (10 clases)",
        font=dict(size=15, color="#e0ddd6", family="Arial"),
        x=0.01,
    ),
    paper_bgcolor="#0d1117",
    plot_bgcolor="#0d1117",
    font=dict(color="#e0ddd6", family="Arial", size=11),
    height=550,
    barmode="stack",
    margin=dict(t=80, b=80, l=60, r=40),
    legend=dict(
        bgcolor="#1a1a1a", bordercolor="#333333", borderwidth=1,
        font=dict(size=10), orientation="v", x=1.02,
    )
)
fig.update_xaxes(gridcolor="#222222", tickfont=dict(size=9))
fig.update_yaxes(gridcolor="#222222", title_text="Registros",   row=1, col=1)
fig.update_yaxes(gridcolor="#222222", title_text="Porcentaje %", row=1, col=2)
fig.show()

# ── Resultados ─────────────────────────────────────────────────
print(f"Total registros : {total}")
print(f"Clases          : {len(orden)}")
print()
print(f"{'Sistema':<38} {'N':>5}  {'%':>6}  Nivel")
print("─" * 60)
for sistema, n in zip(orden, valores):
    pct    = round(n / total * 100, 1)
    nivel  = "⚠ subrepresentada" if pct < 5 else "↑ dominante" if pct > 25 else "✓ aceptable"
    print(f"  {sistema:<36} {n:>5}  {pct:>5}%  {nivel}")

print()
print("Tabla cruzada tipo de equipo × sistema:")
print(pd.crosstab(
    df_modelo["tipo_corto"],
    df_modelo[TARGET_SISTEMA],
    margins=True,
    margins_name="Total"
).to_string())

Total registros : 1198
Clases          : 10

Sistema                                    N       %  Nivel
────────────────────────────────────────────────────────────
  mecánico y de posicionamiento          192   16.0%  ✓ aceptable
  generación y detección de rayos x      178   14.9%  ✓ aceptable
  eléctrico del equipo                   163   13.6%  ✓ aceptable
  comunicaciones                         151   12.6%  ✓ aceptable
  procesamiento y almacenamiento         149   12.4%  ✓ aceptable
  control e interfaz de usuario          142   11.9%  ✓ aceptable
  falla desconocida                       82    6.8%  ✓ aceptable
  accesorios                              61    5.1%  ✓ aceptable
  usuario                                 42    3.5%  ⚠ subrepresentada
  seguridad del paciente                  38    3.2%  ⚠ subrepresentada

Tabla cruzada tipo de equipo × sistema:
sistema_general_target  accesorios  comunicaciones  control e interfaz de usuario  eléctrico del equipo  falla desconocid

19. UMAP del modelo B. Reducción de dimensionalidad no lineal a 2D y 3D del espacio de features del Modelo B, incluyendo componente_agrupado como variable categórica codificada. El UMAP 2D muestra la separabilidad por sistema y por tipo de equipo. El UMAP 3D revela estructura adicional no visible en 2D. El Silhouette Score cuantifica la separación real entre grupos en ambas dimensiones.

In [62]:
# ── CELDA 19 — UMAP MODELO B ──────────────────────────────────
import plotly.graph_objects as go
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import silhouette_score

try:
    import umap
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "umap-learn", "-q"])
    import umap

if "tipo_corto" not in df_modelo.columns:
    df_modelo["tipo_corto"] = df_modelo["tipo_equipo"].map(NOMBRES_CORTOS)

COLORES_SISTEMA = {
    "mecánico y de posicionamiento"      : "#378ADD",
    "generación y detección de rayos x"  : "#EF9F27",
    "eléctrico del equipo"               : "#E24B4A",
    "control e interfaz de usuario"      : "#1D9E75",
    "procesamiento y almacenamiento"     : "#7F77DD",
    "comunicaciones"                     : "#D85A30",
    "falla desconocida"                  : "#888888",
    "accesorios"                         : "#639922",
    "seguridad del paciente"             : "#534AB7",
    "usuario"                            : "#CC6688",
}

COLORES_EQUIPO = {
    "TAC"        : "#378ADD",
    "Fluoro"     : "#639922",
    "RX Fijo"    : "#7F77DD",
    "Mamografía" : "#D85A30",
    "RX Móvil"   : "#1D9E75",
    "Arco C"     : "#BA7517",
    "Angiografía": "#E24B4A",
}

le = LabelEncoder()
df_umap_b = df_modelo.copy()
for col in ["tipo_solucion", "tipo_equipo", "componente_agrupado"]:
    df_umap_b[col + "_enc"] = le.fit_transform(df_umap_b[col].astype(str))

VARS_UMAP_B = [
    "tipo_solucion_enc",
    "hubo_cambio_componente",
    "tipo_equipo_enc",
    "criticidad",
    "fallos_30_dias",
    "edad_dias",
    "tbf_promedio_equipo",
    "tendencia_tbf",
    "horas_parada_acumuladas",
    "componente_agrupado_enc",
]

df_umap_b = df_umap_b[VARS_UMAP_B + [TARGET_SISTEMA, "tipo_equipo"]].dropna().copy()
df_umap_b["tipo_corto"] = df_umap_b["tipo_equipo"].map(NOMBRES_CORTOS)

X_scaled = StandardScaler().fit_transform(df_umap_b[VARS_UMAP_B].values)

print("Calculando UMAP 2D...")
X_2d = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1,
                  random_state=42, verbose=False).fit_transform(X_scaled)
print("UMAP 2D listo")

print("Calculando UMAP 3D...")
X_3d = umap.UMAP(n_components=3, n_neighbors=15, min_dist=0.1,
                  random_state=42, verbose=False).fit_transform(X_scaled)
print("UMAP 3D listo")

# ── Figura 1 — UMAP 2D por sistema ────────────────────────────
fig1 = go.Figure()
for sistema, color in COLORES_SISTEMA.items():
    mask = df_umap_b[TARGET_SISTEMA] == sistema
    if mask.sum() == 0:
        continue
    fig1.add_trace(go.Scatter(
        x=X_2d[mask, 0], y=X_2d[mask, 1],
        mode="markers", name=sistema,
        marker=dict(size=5, color=color, opacity=0.7, line=dict(width=0)),
        hovertemplate=f"<b>{sistema}</b><br>UMAP1: %{{x:.2f}}<br>UMAP2: %{{y:.2f}}<extra></extra>",
    ))
fig1.update_layout(
    title=dict(text="UMAP 2D — Modelo B coloreado por sistema",
               font=dict(size=15, color="#e0ddd6", family="Arial"), x=0.01),
    xaxis_title="UMAP 1", yaxis_title="UMAP 2",
    paper_bgcolor="#0d1117", plot_bgcolor="#0d1117",
    font=dict(color="#e0ddd6", family="Arial", size=11),
    height=520, margin=dict(t=70, b=50, l=60, r=40),
    xaxis=dict(gridcolor="#222222", zerolinecolor="#333333"),
    yaxis=dict(gridcolor="#222222", zerolinecolor="#333333"),
    legend=dict(bgcolor="#1a1a1a", bordercolor="#333333", borderwidth=1),
)
fig1.show()

# ── Figura 2 — UMAP 2D por tipo de equipo ─────────────────────
fig2 = go.Figure()
for tipo, color in COLORES_EQUIPO.items():
    mask = df_umap_b["tipo_corto"] == tipo
    if mask.sum() == 0:
        continue
    fig2.add_trace(go.Scatter(
        x=X_2d[mask, 0], y=X_2d[mask, 1],
        mode="markers", name=tipo,
        marker=dict(size=5, color=color, opacity=0.7, line=dict(width=0)),
        hovertemplate=f"<b>{tipo}</b><br>UMAP1: %{{x:.2f}}<br>UMAP2: %{{y:.2f}}<extra></extra>",
    ))
fig2.update_layout(
    title=dict(text="UMAP 2D — Modelo B coloreado por tipo de equipo",
               font=dict(size=15, color="#e0ddd6", family="Arial"), x=0.01),
    xaxis_title="UMAP 1", yaxis_title="UMAP 2",
    paper_bgcolor="#0d1117", plot_bgcolor="#0d1117",
    font=dict(color="#e0ddd6", family="Arial", size=11),
    height=520, margin=dict(t=70, b=50, l=60, r=40),
    xaxis=dict(gridcolor="#222222", zerolinecolor="#333333"),
    yaxis=dict(gridcolor="#222222", zerolinecolor="#333333"),
    legend=dict(bgcolor="#1a1a1a", bordercolor="#333333", borderwidth=1),
)
fig2.show()

# ── Figura 3 — UMAP 3D por sistema ────────────────────────────
fig3 = go.Figure()
for sistema, color in COLORES_SISTEMA.items():
    mask = df_umap_b[TARGET_SISTEMA] == sistema
    if mask.sum() == 0:
        continue
    fig3.add_trace(go.Scatter3d(
        x=X_3d[mask, 0], y=X_3d[mask, 1], z=X_3d[mask, 2],
        mode="markers", name=sistema,
        marker=dict(size=3, color=color, opacity=0.7, line=dict(width=0)),
        hovertemplate=f"<b>{sistema}</b><br>U1: %{{x:.2f}}<br>U2: %{{y:.2f}}<br>U3: %{{z:.2f}}<extra></extra>",
    ))
fig3.update_layout(
    title=dict(text="UMAP 3D — Modelo B coloreado por sistema",
               font=dict(size=14, color="#e0ddd6", family="Arial"), x=0.01),
    scene=dict(
        xaxis=dict(title="UMAP 1", backgroundcolor="#0d1117", gridcolor="#222222",
                   titlefont=dict(color="#aaaaaa", size=11), tickfont=dict(color="#888888", size=9)),
        yaxis=dict(title="UMAP 2", backgroundcolor="#0d1117", gridcolor="#222222",
                   titlefont=dict(color="#aaaaaa", size=11), tickfont=dict(color="#888888", size=9)),
        zaxis=dict(title="UMAP 3", backgroundcolor="#0d1117", gridcolor="#222222",
                   titlefont=dict(color="#aaaaaa", size=11), tickfont=dict(color="#888888", size=9)),
        bgcolor="#0d1117", camera=dict(eye=dict(x=1.5, y=1.5, z=1.2)),
    ),
    paper_bgcolor="#0d1117",
    font=dict(color="#e0ddd6", family="Arial"),
    legend=dict(bgcolor="#1a1a1a", bordercolor="#333333", borderwidth=1, font=dict(size=11)),
    height=650, margin=dict(t=60, b=20, l=20, r=20),
)
fig3.show()

# ── Silhouette ─────────────────────────────────────────────────
sistemas_unicos = {s: i for i, s in enumerate(COLORES_SISTEMA.keys())}
equipos_unicos  = {e: i for i, e in enumerate(COLORES_EQUIPO.keys())}

labels_sis = df_umap_b[TARGET_SISTEMA].map(sistemas_unicos)
labels_equ = df_umap_b["tipo_corto"].map(equipos_unicos)
mask_val   = labels_sis.notna() & labels_equ.notna()

sil_sis_2d = silhouette_score(X_2d[mask_val], labels_sis[mask_val])
sil_equ_2d = silhouette_score(X_2d[mask_val], labels_equ[mask_val])
sil_sis_3d = silhouette_score(X_3d[mask_val], labels_sis[mask_val])
sil_equ_3d = silhouette_score(X_3d[mask_val], labels_equ[mask_val])

def nivel_sil(s):
    if s > 0.5:   return "Separación excelente"
    if s > 0.3:   return "Separación buena"
    if s > 0.1:   return "Separación moderada"
    return "Grupos muy mezclados"

# ── Resultados ─────────────────────────────────────────────────
print(f"{'Agrupación':<25} {'2D':>10} {'3D':>10}  Interpretación")
print("─" * 65)
for nombre, s2d, s3d in [
    ("Por sistema",        sil_sis_2d, sil_sis_3d),
    ("Por tipo de equipo", sil_equ_2d, sil_equ_3d),
]:
    print(f"  {nombre:<23} {s2d:>10.3f} {s3d:>10.3f}  {nivel_sil(s3d)}")

print(f"\nCentros por sistema en espacio UMAP 2D:")
for sistema in COLORES_SISTEMA.keys():
    mask = df_umap_b[TARGET_SISTEMA] == sistema
    if mask.sum() > 0:
        cx = X_2d[mask, 0].mean()
        cy = X_2d[mask, 1].mean()
        print(f"  {sistema:<35} centro: ({cx:.2f}, {cy:.2f})")

Calculando UMAP 2D...


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



UMAP 2D listo
Calculando UMAP 3D...


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



UMAP 3D listo


Agrupación                        2D         3D  Interpretación
─────────────────────────────────────────────────────────────────
  Por sistema                 -0.183     -0.187  Grupos muy mezclados
  Por tipo de equipo          -0.097     -0.105  Grupos muy mezclados

Centros por sistema en espacio UMAP 2D:
  mecánico y de posicionamiento       centro: (5.88, 8.71)
  generación y detección de rayos x   centro: (7.23, 8.58)
  eléctrico del equipo                centro: (5.28, 7.97)
  control e interfaz de usuario       centro: (5.90, 7.28)
  procesamiento y almacenamiento      centro: (8.02, 6.45)
  comunicaciones                      centro: (9.79, 7.80)
  falla desconocida                   centro: (8.89, 6.15)
  accesorios                          centro: (5.20, 5.73)
  seguridad del paciente              centro: (7.21, 6.65)
  usuario                             centro: (8.68, 5.98)


# Bloque 3 — Modelado y entrenamiento.

Con el dataset limpio, las features construidas y el análisis exploratorio completado, se procede al entrenamiento de los modelos predictivos. Este bloque implementa dos modelos XGBoost independientes: el Modelo A predice el periodo de fallo (Alta urgencia, Programable, Preventivo) y el Modelo B predice el sistema afectado (10 clases).

Antes del entrenamiento se establece un baseline con DummyClassifier — un modelo trivial que predice siempre la clase más frecuente sin considerar ninguna feature. Este baseline es el punto de referencia mínimo que cualquier modelo útil debe superar. Si XGBoost no mejorara significativamente sobre el baseline, no tendría justificación técnica su uso.

El desbalanceo de clases se maneja mediante compute_sample_weight("balanced"), que asigna mayor peso a las clases minoritarias durante el entrenamiento sin modificar la distribución real del dataset — a diferencia de técnicas de sobremuestreo como SMOTE que generan datos sintéticos.

Los resultados del EDA mostraron que las clases no presentan separabilidad lineal ni geométrica en ninguno de los dos modelos — Silhouette negativo en UMAP 2D y 3D para ambos targets. Esto motiva el uso de XGBoost, capaz de encontrar fronteras de decisión no lineales complejas mediante particiones sucesivas del espacio de features que técnicas más simples no pueden capturar.

20. Baseline-DummyClassifier. Se establece el piso mínimo de rendimiento usando un DummyClassifier con estrategia most_frequent — predice siempre la clase más frecuente independientemente de las features. El F1-macro del baseline es bajo porque penaliza las clases minoritarias con F1 de 0. Todo modelo que aprenda patrones reales debe superar estos valores significativamente. Los splits y encoders son idénticos a los de los modelos finales para garantizar comparabilidad.

In [63]:
# ── CELDA 20 — BASELINE (DummyClassifier) ─────────────────────
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.metrics import f1_score

# ── Baseline Modelo A ──────────────────────────────────────────
FEATURES_A     = [
    "fallos_30_dias", "tbf_promedio_equipo", "tendencia_tbf",
    "criticidad", "edad_dias", "horas_parada_acumuladas",
    "tasa_fallo_equipo", "dias_desde_ultimo_fallo",
    "tipo_solucion", "hubo_cambio_componente", "tipo_equipo",
]
FEATURES_A_CAT = ["tipo_solucion", "tipo_equipo"]

df_base_a = df_modelo[FEATURES_A + [TARGET_PERIODO]].dropna().copy()
enc_base_a = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
df_base_a[FEATURES_A_CAT] = enc_base_a.fit_transform(df_base_a[FEATURES_A_CAT])
le_base_a  = LabelEncoder()
df_base_a[TARGET_PERIODO] = le_base_a.fit_transform(df_base_a[TARGET_PERIODO])

X_a = df_base_a[FEATURES_A].values
y_a = df_base_a[TARGET_PERIODO].values
X_a_train, X_a_test, y_a_train, y_a_test = train_test_split(
    X_a, y_a, test_size=0.20, random_state=42, stratify=y_a
)

dummy_a      = DummyClassifier(strategy="most_frequent", random_state=42)
dummy_a.fit(X_a_train, y_a_train)
f1_dummy_a   = f1_score(y_a_test, dummy_a.predict(X_a_test), average="macro")
pred_a       = dummy_a.predict([X_a_test[0]])[0]
clase_pred_a = le_base_a.classes_[pred_a]

# ── Baseline Modelo B ──────────────────────────────────────────
FEATURES_B     = [
    "tipo_solucion", "hubo_cambio_componente", "tipo_equipo",
    "criticidad", "fallos_30_dias", "edad_dias",
    "tbf_promedio_equipo", "tendencia_tbf",
    "horas_parada_acumuladas", "componente_agrupado",
]
FEATURES_B_CAT = ["tipo_solucion", "tipo_equipo", "componente_agrupado"]

df_base_b = df_modelo[FEATURES_B + [TARGET_SISTEMA]].dropna().copy()
enc_base_b = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
df_base_b[FEATURES_B_CAT] = enc_base_b.fit_transform(df_base_b[FEATURES_B_CAT])
le_base_b  = LabelEncoder()
df_base_b[TARGET_SISTEMA] = le_base_b.fit_transform(df_base_b[TARGET_SISTEMA])

X_b = df_base_b[FEATURES_B].values
y_b = df_base_b[TARGET_SISTEMA].values
X_b_train, X_b_test, y_b_train, y_b_test = train_test_split(
    X_b, y_b, test_size=0.20, random_state=42, stratify=y_b
)

dummy_b      = DummyClassifier(strategy="most_frequent", random_state=42)
dummy_b.fit(X_b_train, y_b_train)
f1_dummy_b   = f1_score(y_b_test, dummy_b.predict(X_b_test), average="macro")
pred_b       = dummy_b.predict([X_b_test[0]])[0]
clase_pred_b = le_base_b.classes_[pred_b]

# ── Resultados ─────────────────────────────────────────────────
print("  {:<10} {:<20} {:<35} {:>10}".format("Modelo A", "most_frequent", clase_pred_a, round(f1_dummy_a, 3)))
print("  {:<10} {:<20} {:<35} {:>10}".format("Modelo B", "most_frequent", clase_pred_b, round(f1_dummy_b, 3)))
print()
print("  El baseline predice siempre la clase mas frecuente.")
print("  Cualquier modelo util debe superar ampliamente estos valores.")

  Modelo A   most_frequent        Programable                              0.211
  Modelo B   most_frequent        mecánico y de posicionamiento            0.028

  El baseline predice siempre la clase mas frecuente.
  Cualquier modelo util debe superar ampliamente estos valores.


21. Modelo A- Periodo_fallo.
Se entrena el Modelo A con XGBoost para clasificar el periodo de fallo en tres clases operacionales. Las features de frecuencia de fallos en 30 y 90 días concentran el 77% de la importancia del modelo, confirmando que el ritmo reciente de fallos es la señal más predictiva de urgencia. Se aplica compute_sample_weight("balanced") para compensar el desbalanceo de la clase Preventivo. El modelo alcanza F1-macro de **0.872** con gap train-test de 0.003, sin sobreentrenamiento. Al finalizar se exportan el modelo, el encoder y el mapa de clases.

In [64]:
# ── CELDA 21 — MODELO A ───────────────────────────────────────
import subprocess
subprocess.run(["pip", "install", "kaleido==0.2.1", "--quiet"], capture_output=True)
import kaleido
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.preprocessing import OrdinalEncoder
from sklearn.utils.class_weight import compute_sample_weight
import plotly.graph_objects as go
import joblib, os

os.makedirs("/content/artefactos", exist_ok=True)

# ── 1. Feature set ────────────────────────────────────────────
FEATURES_A = [
    "fallos_30_dias",
    "tbf_promedio_equipo",
    "tendencia_tbf",
    "criticidad",
    "edad_dias",
    "horas_parada_acumuladas",
    "fallos_90_dias",
    "tasa_fallo_equipo",
    "dias_desde_ultimo_fallo",
    "tipo_equipo",
]
FEATURES_A_CAT = ["tipo_equipo"]

NOMBRES_A = {
    "fallos_30_dias"          : "Fallos 30 días",
    "tbf_promedio_equipo"     : "TBF promedio equipo",
    "tendencia_tbf"           : "Tendencia TBF",
    "criticidad"              : "Criticidad",
    "edad_dias"               : "Edad (días)",
    "horas_parada_acumuladas" : "Horas parada acum.",
    "fallos_90_dias"          : "Fallos 90 días",
    "tasa_fallo_equipo"       : "Tasa fallo equipo",
    "dias_desde_ultimo_fallo" : "Días desde último fallo",
    "tipo_equipo"             : "Tipo equipo",
}

MAPA_PERIODO = {"Alta urgencia": 0, "Programable": 1, "Preventivo": 2}
clases_a     = ["Alta urgencia", "Programable", "Preventivo"]
COLORES_3    = ["#E24B4A", "#378ADD", "#1D9E75"]

# ── 2. Preparar X e y ─────────────────────────────────────────
df_a = df_modelo[FEATURES_A + [TARGET_PERIODO]].dropna().copy()

enc_a = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
df_a[FEATURES_A_CAT] = enc_a.fit_transform(df_a[FEATURES_A_CAT])

df_a[TARGET_PERIODO] = df_a[TARGET_PERIODO].map(MAPA_PERIODO)

X_a = df_a[FEATURES_A].values
y_a = df_a[TARGET_PERIODO].values

X_a_train, X_a_test, y_a_train, y_a_test = train_test_split(
    X_a, y_a,
    test_size=0.20,
    random_state=42,
    stratify=y_a,
)

pesos_a = compute_sample_weight("balanced", y_a_train)

print("Split Modelo A:")
print("  Train : {} registros".format(len(X_a_train)))
print("  Test  : {} registros".format(len(X_a_test)))
print("  Clases: {}".format(len(np.unique(y_a_train))))

# ── 3. Entrenar ───────────────────────────────────────────────
modelo_a = XGBClassifier(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.7,
    colsample_bytree=0.7,
    min_child_weight=5,
    reg_alpha=0.1,
    reg_lambda=2.0,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1,
)
modelo_a.fit(X_a_train, y_a_train, sample_weight=pesos_a)

y_a_pred       = modelo_a.predict(X_a_test)
y_a_pred_train = modelo_a.predict(X_a_train)

f1_a       = f1_score(y_a_test,  y_a_pred,       average="macro")
f1_a_train = f1_score(y_a_train, y_a_pred_train, average="macro")
f1_a_clase = f1_score(y_a_test,  y_a_pred,       average=None)
cm_a       = confusion_matrix(y_a_test, y_a_pred)
cm_a_norm  = cm_a.astype(float) / cm_a.sum(axis=1, keepdims=True)

# ── 4. Verificación sobreentrenamiento ────────────────────────
gap_a = f1_a_train - f1_a
print("=" * 55)
print("  Verificación sobreentrenamiento — Modelo A")
print("=" * 55)
print("  F1-macro TRAIN : {:.3f}".format(f1_a_train))
print("  F1-macro TEST  : {:.3f}".format(f1_a))
print("  Gap train-test : {:.3f}".format(gap_a))
if gap_a > 0.10:
    print("  ⚠ Gap > 0.10 → sobreentrenamiento")
elif gap_a > 0.05:
    print("  ⚡ Gap 0.05-0.10 → leve, aceptable")
else:
    print("  ✓ Gap < 0.05 → modelo generaliza bien")

# ── 5. Matriz de confusión ────────────────────────────────────
n = len(clases_a)
fig1 = go.Figure(go.Heatmap(
    z=cm_a_norm,
    x=clases_a,
    y=clases_a,
    colorscale=[[0, "#0d1117"], [1, "#1D9E75"]],
    text=[[str(cm_a[i][j]) for j in range(n)] for i in range(n)],
    texttemplate="%{text}",
    textfont=dict(size=14, color="#ffffff"),
    showscale=True, zmin=0, zmax=1,
))
fig1.update_layout(
    title=dict(text="Matriz de confusión — Modelo A",
               font=dict(size=14, color="#e0ddd6", family="Arial"), x=0.01),
    xaxis=dict(title="Predicho", tickfont=dict(color="#e0ddd6"), gridcolor="#222222"),
    yaxis=dict(title="Real", tickfont=dict(color="#e0ddd6"),
               gridcolor="#222222", autorange="reversed"),
    paper_bgcolor="#0d1117", plot_bgcolor="#0d1117",
    font=dict(color="#e0ddd6", family="Arial"),
    height=380, margin=dict(t=60, b=60, l=130, r=40),
)
fig1.show()

# ── 6. F1 por clase ───────────────────────────────────────────
fig2 = go.Figure(go.Bar(
    x=clases_a,
    y=f1_a_clase,
    marker_color=COLORES_3,
    marker_line_width=0,
    text=["{:.3f}".format(v) for v in f1_a_clase],
    textposition="outside",
    textfont=dict(size=11, color="#aaaaaa"),
))
fig2.add_hline(
    y=f1_a, line_dash="dot", line_color="#ffffff", line_width=1,
    annotation_text="F1-macro: {:.3f}".format(f1_a),
    annotation_font_color="#ffffff", annotation_position="top right",
)
fig2.update_layout(
    title=dict(text="F1 por clase — Modelo A",
               font=dict(size=14, color="#e0ddd6", family="Arial"), x=0.01),
    xaxis=dict(tickfont=dict(color="#e0ddd6", size=11), gridcolor="#222222"),
    yaxis=dict(gridcolor="#222222", range=[0, 1.1],
               title="F1-score", tickfont=dict(color="#e0ddd6")),
    paper_bgcolor="#0d1117", plot_bgcolor="#0d1117",
    font=dict(color="#e0ddd6", family="Arial"),
    height=400, margin=dict(t=60, b=60, l=60, r=40),
)
fig2.show()

# ── 7. Feature importance ─────────────────────────────────────
imp_a     = modelo_a.feature_importances_
orden_a   = np.argsort(imp_a)[::-1]
names_ord = [NOMBRES_A.get(FEATURES_A[i], FEATURES_A[i]) for i in orden_a]
imp_ord   = imp_a[orden_a]

COLORES_IMP = ["#1D9E75" if v >= 0.20 else "#EF9F27" if v >= 0.10 else "#378ADD"
               for v in imp_ord]

fig3 = go.Figure(go.Bar(
    x=imp_ord, y=names_ord,
    orientation="h",
    marker_color=COLORES_IMP, marker_line_width=0,
    text=["{:.3f}".format(v) for v in imp_ord],
    textposition="outside",
    textfont=dict(size=11, color="#aaaaaa"),
))
fig3.update_layout(
    title=dict(text="Feature importance — Modelo A",
               font=dict(size=14, color="#e0ddd6", family="Arial"), x=0.01),
    xaxis=dict(title="Importancia relativa", gridcolor="#222222",
               range=[0, max(imp_ord) * 1.2], tickfont=dict(color="#e0ddd6")),
    yaxis=dict(gridcolor="#222222", tickfont=dict(color="#e0ddd6", size=12),
               autorange="reversed"),
    paper_bgcolor="#0d1117", plot_bgcolor="#0d1117",
    font=dict(color="#e0ddd6", family="Arial"),
    height=450, margin=dict(t=60, b=60, l=180, r=80),
)
fig3.show()

# ── 8. Exportar artefactos ────────────────────────────────────
modelo_a.save_model("/content/artefactos/modelo_a.ubj")
joblib.dump(enc_a,        "/content/artefactos/enc_a.pkl")
joblib.dump(MAPA_PERIODO, "/content/artefactos/mapa_periodo.pkl")

# ── 9. Resumen ────────────────────────────────────────────────
print("=" * 55)
print("  MODELO A — resultado final")
print("=" * 55)
print("  F1-macro TEST  : {:.3f}".format(f1_a))
print("  F1-macro TRAIN : {:.3f}".format(f1_a_train))
print("  Gap            : {:.3f}".format(gap_a))
print("  Baseline       : {:.3f}  (+{:.3f} mejora)".format(
    f1_dummy_a, f1_a - f1_dummy_a))
print()
print("  F1 por clase:")
for nombre, val in zip(clases_a, f1_a_clase):
    barra = "█" * int(val * 30)
    print("    {:<22} {:.3f}  {}".format(nombre, val, barra))
print()
print(classification_report(
    y_a_test, y_a_pred,
    target_names=clases_a,
    digits=3,
))
print("  Artefactos exportados en /content/artefactos/")
print("    modelo_a.ubj")
print("    enc_a.pkl")
print("    mapa_periodo.pkl")

Split Modelo A:
  Train : 958 registros
  Test  : 240 registros
  Clases: 3
  Verificación sobreentrenamiento — Modelo A
  F1-macro TRAIN : 0.875
  F1-macro TEST  : 0.872
  Gap train-test : 0.003
  ✓ Gap < 0.05 → modelo generaliza bien


  MODELO A — resultado final
  F1-macro TEST  : 0.872
  F1-macro TRAIN : 0.875
  Gap            : 0.003
  Baseline       : 0.211  (+0.661 mejora)

  F1 por clase:
    Alta urgencia          0.829  ████████████████████████
    Programable            0.825  ████████████████████████
    Preventivo             0.962  ████████████████████████████

               precision    recall  f1-score   support

Alta urgencia      0.780     0.885     0.829       104
  Programable      0.895     0.766     0.825       111
   Preventivo      0.926     1.000     0.962        25

     accuracy                          0.842       240
    macro avg      0.867     0.883     0.872       240
 weighted avg      0.848     0.842     0.841       240

  Artefactos exportados en /content/artefactos/
    modelo_a.ubj
    enc_a.pkl
    mapa_periodo.pkl


22. Modelo B — Sistema afectado (sistema_general_target)

Se entrena un clasificador XGBoost para predecir el sistema afectado
en cada intervención de mantenimiento correctivo, sobre 10 clases operacionales.
Emplea 10 features de entrada: 3 categóricas codificadas con OrdinalEncoder
(tipo_solucion, tipo_equipo, componente_agrupado) y 7 numéricas.
El desbalance de clases se maneja con compute_sample_weight("balanced").
El target sistema_general_target es codificado con LabelEncoder (le_b).


In [65]:
# ── CELDA 22 — MODELO B ───────────────────────────────────────
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
import plotly.graph_objects as go
import joblib

# ── 1. Feature set ────────────────────────────────────────────
FEATURES_B = [
    "tipo_solucion",
    "hubo_cambio_componente",
    "tipo_equipo",
    "criticidad",
    "fallos_30_dias",
    "edad_dias",
    "tbf_promedio_equipo",
    "tendencia_tbf",
    "horas_parada_acumuladas",
    "componente_agrupado",
]
FEATURES_B_CAT = ["tipo_solucion", "tipo_equipo", "componente_agrupado"]

NOMBRES_B = {
    "tipo_solucion"           : "Tipo solución",
    "hubo_cambio_componente"  : "Cambio componente",
    "tipo_equipo"             : "Tipo equipo",
    "criticidad"              : "Criticidad",
    "fallos_30_dias"          : "Fallos 30 días",
    "edad_dias"               : "Edad (días)",
    "tbf_promedio_equipo"     : "TBF promedio equipo",
    "tendencia_tbf"           : "Tendencia TBF",
    "horas_parada_acumuladas" : "Horas parada acum.",
    "componente_agrupado"     : "Componente agrupado",
}

TARGET_B = "sistema_general_target"

# ── 2. Verificar target ───────────────────────────────────────
print("Distribución sistema_general_target:")
print(df_modelo[TARGET_B].value_counts())
print("\nTotal clases: " + str(df_modelo[TARGET_B].nunique()))

# ── 3. Preparar X e y ─────────────────────────────────────────
df_b = df_modelo[FEATURES_B + [TARGET_B]].dropna().copy()

enc_b = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
df_b[FEATURES_B_CAT] = enc_b.fit_transform(df_b[FEATURES_B_CAT])

le_b = LabelEncoder()
df_b[TARGET_B] = le_b.fit_transform(df_b[TARGET_B])

X_b = df_b[FEATURES_B].values
y_b = df_b[TARGET_B].values

X_b_train, X_b_test, y_b_train, y_b_test = train_test_split(
    X_b, y_b,
    test_size=0.20,
    random_state=42,
    stratify=y_b,
)

pesos_b = compute_sample_weight("balanced", y_b_train)

print("\nSplit Modelo B:")
print("  Train: {} registros".format(len(X_b_train)))
print("  Test:  {} registros".format(len(X_b_test)))
print("  Clases: {}".format(len(np.unique(y_b_train))))

# ── 4. Entrenar ───────────────────────────────────────────────
modelo_b = XGBClassifier(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.7,
    colsample_bytree=0.7,
    min_child_weight=5,
    reg_alpha=0.1,
    reg_lambda=2.0,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1,
)
modelo_b.fit(X_b_train, y_b_train, sample_weight=pesos_b)

y_b_pred       = modelo_b.predict(X_b_test)
y_b_pred_train = modelo_b.predict(X_b_train)

f1_b       = f1_score(y_b_test,  y_b_pred,       average="macro")
f1_b_train = f1_score(y_b_train, y_b_pred_train, average="macro")
f1_b_clase = f1_score(y_b_test,  y_b_pred,       average=None)
cm_b       = confusion_matrix(y_b_test, y_b_pred)
cm_b_norm  = cm_b.astype(float) / cm_b.sum(axis=1, keepdims=True)
clases_b   = le_b.classes_

# ── 5. Verificación sobreentrenamiento ────────────────────────
gap_b = f1_b_train - f1_b
print("=" * 55)
print("  Verificación sobreentrenamiento — Modelo B")
print("=" * 55)
print("  F1-macro TRAIN : {:.3f}".format(f1_b_train))
print("  F1-macro TEST  : {:.3f}".format(f1_b))
print("  Gap train-test : {:.3f}".format(gap_b))
if gap_b > 0.10:
    print("  ⚠ Gap > 0.10 → sobreentrenamiento")
elif gap_b > 0.05:
    print("  ⚡ Gap 0.05-0.10 → leve, aceptable")
else:
    print("  ✓ Gap < 0.05 → modelo generaliza bien")

# ── 6. Matriz de confusión ────────────────────────────────────
n = len(clases_b)
fig1 = go.Figure(go.Heatmap(
    z=cm_b_norm,
    x=clases_b.tolist(),
    y=clases_b.tolist(),
    colorscale=[[0, "#0d1117"], [1, "#534AB7"]],
    text=[[str(cm_b[i][j]) for j in range(n)] for i in range(n)],
    texttemplate="%{text}",
    textfont=dict(size=10, color="#ffffff"),
    showscale=True, zmin=0, zmax=1,
))
fig1.update_layout(
    title=dict(text="Matriz de confusión — Modelo B",
               font=dict(size=14, color="#e0ddd6", family="Arial"), x=0.01),
    xaxis=dict(title="Predicho", tickfont=dict(color="#e0ddd6", size=9),
               tickangle=-35, gridcolor="#222222"),
    yaxis=dict(title="Real", tickfont=dict(color="#e0ddd6", size=9),
               gridcolor="#222222", autorange="reversed"),
    paper_bgcolor="#0d1117", plot_bgcolor="#0d1117",
    font=dict(color="#e0ddd6", family="Arial"),
    height=550, margin=dict(t=60, b=130, l=160, r=40),
)
fig1.show()

# ── 7. F1 por clase ───────────────────────────────────────────
COLORES_B = ["#534AB7" if v >= 0.20 else "#7F77DD" if v >= 0.10 else "#AFA9EC"
             for v in f1_b_clase]

fig2 = go.Figure(go.Bar(
    x=clases_b.tolist(),
    y=f1_b_clase,
    marker_color=COLORES_B,
    marker_line_width=0,
    text=["{:.3f}".format(v) for v in f1_b_clase],
    textposition="outside",
    textfont=dict(size=10, color="#aaaaaa"),
))
fig2.add_hline(
    y=f1_b, line_dash="dot", line_color="#ffffff", line_width=1,
    annotation_text="F1-macro: {:.3f}".format(f1_b),
    annotation_font_color="#ffffff", annotation_position="top right",
)
fig2.update_layout(
    title=dict(text="F1 por clase — Modelo B",
               font=dict(size=14, color="#e0ddd6", family="Arial"), x=0.01),
    xaxis=dict(tickfont=dict(color="#e0ddd6", size=10),
               tickangle=-30, gridcolor="#222222"),
    yaxis=dict(gridcolor="#222222", range=[0, 1.1],
               title="F1-score", tickfont=dict(color="#e0ddd6")),
    paper_bgcolor="#0d1117", plot_bgcolor="#0d1117",
    font=dict(color="#e0ddd6", family="Arial"),
    height=430, margin=dict(t=60, b=100, l=60, r=40),
)
fig2.show()

# ── 8. Feature importance ─────────────────────────────────────
imp_b     = modelo_b.feature_importances_
orden_b   = np.argsort(imp_b)[::-1]
names_ord = [NOMBRES_B.get(FEATURES_B[i], FEATURES_B[i]) for i in orden_b]
imp_ord   = imp_b[orden_b]

COLORES_IMP = ["#534AB7" if v >= 0.20 else "#7F77DD" if v >= 0.10 else "#AFA9EC"
               for v in imp_ord]

fig3 = go.Figure(go.Bar(
    x=imp_ord, y=names_ord,
    orientation="h",
    marker_color=COLORES_IMP, marker_line_width=0,
    text=["{:.3f}".format(v) for v in imp_ord],
    textposition="outside",
    textfont=dict(size=11, color="#aaaaaa"),
))
fig3.update_layout(
    title=dict(text="Feature importance — Modelo B",
               font=dict(size=14, color="#e0ddd6", family="Arial"), x=0.01),
    xaxis=dict(title="Importancia relativa", gridcolor="#222222",
               range=[0, max(imp_ord) * 1.2], tickfont=dict(color="#e0ddd6")),
    yaxis=dict(gridcolor="#222222", tickfont=dict(color="#e0ddd6", size=12),
               autorange="reversed"),
    paper_bgcolor="#0d1117", plot_bgcolor="#0d1117",
    font=dict(color="#e0ddd6", family="Arial"),
    height=420, margin=dict(t=60, b=60, l=160, r=80),
)
fig3.show()

# ── 9. Exportar artefactos ────────────────────────────────────
modelo_b.save_model("/content/artefactos/modelo_b.ubj")
joblib.dump(enc_b, "/content/artefactos/enc_b.pkl")
joblib.dump(le_b,  "/content/artefactos/le_b.pkl")

# ── 10. Resumen ───────────────────────────────────────────────
print("=" * 58)
print("  MODELO B — resultado final")
print("=" * 58)
print("  F1-macro TEST  : {:.3f}".format(f1_b))
print("  F1-macro TRAIN : {:.3f}".format(f1_b_train))
print("  Gap            : {:.3f}".format(gap_b))
print("  Baseline       : {:.3f}  (+{:.3f} mejora)".format(
    f1_dummy_b, f1_b - f1_dummy_b))
print()
print("  F1 por clase:")
for clase, val in sorted(zip(clases_b, f1_b_clase),
                         key=lambda x: x[1], reverse=True):
    barra = "█" * int(val * 30)
    print("    {:<35} {:.3f}  {}".format(str(clase), val, barra))
print()
print(classification_report(
    y_b_test, y_b_pred,
    target_names=clases_b,
    digits=3,
))
print("  Artefactos exportados en /content/artefactos/")
print("    modelo_b.ubj")
print("    enc_b.pkl")
print("    le_b.pkl")

Distribución sistema_general_target:
sistema_general_target
mecánico y de posicionamiento        192
generación y detección de rayos x    178
eléctrico del equipo                 163
comunicaciones                       151
procesamiento y almacenamiento       149
control e interfaz de usuario        142
falla desconocida                     82
accesorios                            61
usuario                               42
seguridad del paciente                38
Name: count, dtype: int64

Total clases: 10

Split Modelo B:
  Train: 958 registros
  Test:  240 registros
  Clases: 10
  Verificación sobreentrenamiento — Modelo B
  F1-macro TRAIN : 0.956
  F1-macro TEST  : 0.910
  Gap train-test : 0.046
  ✓ Gap < 0.05 → modelo generaliza bien


  MODELO B — resultado final
  F1-macro TEST  : 0.910
  F1-macro TRAIN : 0.956
  Gap            : 0.046
  Baseline       : 0.028  (+0.882 mejora)

  F1 por clase:
    seguridad del paciente              1.000  ██████████████████████████████
    comunicaciones                      0.966  ████████████████████████████
    procesamiento y almacenamiento      0.966  ████████████████████████████
    mecánico y de posicionamiento       0.946  ████████████████████████████
    accesorios                          0.923  ███████████████████████████
    control e interfaz de usuario       0.885  ██████████████████████████
    falla desconocida                   0.882  ██████████████████████████
    usuario                             0.875  ██████████████████████████
    generación y detección de rayos x   0.850  █████████████████████████
    eléctrico del equipo                0.807  ████████████████████████

                                   precision    recall  f1-score   support

            

# Bloque 4 - Exportación de artefactos y visualizaciones

Una vez entrenados y validados ambos modelos, este bloque consolida
la descarga de todos los archivos generados durante el pipeline.

23. a

Descarga los artefactos del modelo: los dos clasificadores
en formato nativo XGBoost (.ubj), los encoders de features categóricas
(enc_a.pkl, enc_b.pkl), el LabelEncoder del target de Modelo B (le_b.pkl)
y el mapa de clases de Modelo A (mapa_periodo.pkl). Estos archivos permiten
reproducir predicciones sin reentrenar.

In [66]:
# CELDA 23a DESCARGA DE ARTEFACTOS DEL MODELO ─────────────────
from google.colab import files
import os

RUTA = "/content/artefactos"

artefactos = [
    "modelo_a.ubj",
    "modelo_b.ubj",
    "enc_a.pkl",
    "enc_b.pkl",
    "le_b.pkl",
    "mapa_periodo.pkl",
]

print("Descargando artefactos del modelo...")
print()
for nombre in artefactos:
    ruta_completa = os.path.join(RUTA, nombre)
    if os.path.exists(ruta_completa):
        files.download(ruta_completa)
        print("✓ {}".format(nombre))
    else:
        print("✗ {} — no encontrado, verifica que celdas 21 y 22 corrieron".format(nombre))

print()
print("Listo. Archivos descargados: {}".format(
    sum(1 for n in artefactos if os.path.exists(os.path.join(RUTA, n)))
))

Descargando artefactos del modelo...



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ modelo_a.ubj


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

✓ le_b.pkl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ mapa_periodo.pkl

Listo. Archivos descargados: 6


23. b

Exporta las matrices de confusión y gráficas de feature
importance de ambos modelos en formato PNG con fondo oscuro, usando
matplotlib para garantizar compatibilidad en este entorno Colab
(kaleido no disponible).

In [67]:
# ── CELDA 23b — EXPORTAR Y DESCARGAR IMÁGENES ──────────────────
import matplotlib.pyplot as plt
import numpy as np
import os
from google.colab import files

os.makedirs("/content/artefactos", exist_ok=True)
plt.style.use("dark_background")
BG = "#0d1117"

def exportar_y_descargar(fig, path):
    plt.savefig(path, dpi=150, bbox_inches="tight", facecolor=BG)
    plt.close()
    files.download(path)
    print("✓ descargado: {}".format(path.split("/")[-1]))

# ── Matriz confusión Modelo A ──────────────────────────────────
n_a = len(clases_a)
fig, ax = plt.subplots(figsize=(7, 5))
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)
im = ax.imshow(cm_a_norm, cmap="Greens", vmin=0, vmax=1)
ax.set_xticks(range(n_a))
ax.set_yticks(range(n_a))
ax.set_xticklabels(clases_a, color="#e0ddd6", fontsize=11)
ax.set_yticklabels(clases_a, color="#e0ddd6", fontsize=11)
ax.set_xlabel("Predicho", color="#e0ddd6", labelpad=10)
ax.set_ylabel("Real", color="#e0ddd6", labelpad=10)
ax.set_title("Matriz de confusión — Modelo A", color="#e0ddd6", pad=15, fontsize=13)
for i in range(n_a):
    for j in range(n_a):
        valor = cm_a[i][j]
        color_texto = "white" if cm_a_norm[i][j] > 0.4 else "#aaaaaa"
        ax.text(j, i, str(valor), ha="center", va="center",
                color=color_texto, fontsize=13, fontweight="bold")
plt.colorbar(im, ax=ax)
plt.tight_layout()
exportar_y_descargar(fig, "/content/artefactos/matriz_confusion_modelo_a.png")

# ── Feature importance Modelo A ────────────────────────────────
idx_ord_a = np.argsort(modelo_a.feature_importances_)
imp_a_ord = modelo_a.feature_importances_[idx_ord_a]
names_a   = [NOMBRES_A.get(FEATURES_A[i], FEATURES_A[i]) for i in idx_ord_a]
colores_a = ["#1D9E75" if v >= 0.20 else "#EF9F27" if v >= 0.10 else "#378ADD"
             for v in imp_a_ord]

fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)
bars = ax.barh(names_a, imp_a_ord, color=colores_a)
for bar, val in zip(bars, imp_a_ord):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
            "{:.3f}".format(val), va="center", color="#aaaaaa", fontsize=10)
ax.set_xlabel("Importancia relativa", color="#e0ddd6")
ax.set_title("Feature importance — Modelo A", color="#e0ddd6", pad=12)
ax.tick_params(colors="#e0ddd6")
ax.spines[:].set_color("#333333")
plt.tight_layout()
exportar_y_descargar(fig, "/content/artefactos/feature_importance_modelo_a.png")

# ── Matriz confusión Modelo B ──────────────────────────────────
n_b = len(clases_b)
fig, ax = plt.subplots(figsize=(12, 9))
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)
im = ax.imshow(cm_b_norm, cmap="Purples", vmin=0, vmax=1)
ax.set_xticks(range(n_b))
ax.set_yticks(range(n_b))
ax.set_xticklabels(clases_b.tolist(), color="#e0ddd6", fontsize=9,
                   rotation=35, ha="right")
ax.set_yticklabels(clases_b.tolist(), color="#e0ddd6", fontsize=9)
ax.set_xlabel("Predicho", color="#e0ddd6", labelpad=10)
ax.set_ylabel("Real", color="#e0ddd6", labelpad=10)
ax.set_title("Matriz de confusión — Modelo B", color="#e0ddd6", pad=15, fontsize=13)
for i in range(n_b):
    for j in range(n_b):
        valor = cm_b[i][j]
        color_texto = "white" if cm_b_norm[i][j] > 0.4 else "#aaaaaa"
        ax.text(j, i, str(valor), ha="center", va="center",
                color=color_texto, fontsize=9, fontweight="bold")
plt.colorbar(im, ax=ax)
plt.tight_layout()
exportar_y_descargar(fig, "/content/artefactos/matriz_confusion_modelo_b.png")

# ── Feature importance Modelo B ────────────────────────────────
idx_ord_b = np.argsort(modelo_b.feature_importances_)
imp_b_ord = modelo_b.feature_importances_[idx_ord_b]
names_b   = [NOMBRES_B.get(FEATURES_B[i], FEATURES_B[i]) for i in idx_ord_b]
colores_b = ["#534AB7" if v >= 0.20 else "#7F77DD" if v >= 0.10 else "#AFA9EC"
             for v in imp_b_ord]

fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)
bars = ax.barh(names_b, imp_b_ord, color=colores_b)
for bar, val in zip(bars, imp_b_ord):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
            "{:.3f}".format(val), va="center", color="#aaaaaa", fontsize=10)
ax.set_xlabel("Importancia relativa", color="#e0ddd6")
ax.set_title("Feature importance — Modelo B", color="#e0ddd6", pad=12)
ax.tick_params(colors="#e0ddd6")
ax.spines[:].set_color("#333333")
plt.tight_layout()
exportar_y_descargar(fig, "/content/artefactos/feature_importance_modelo_b.png")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ descargado: matriz_confusion_modelo_a.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

✓ descargado: matriz_confusion_modelo_b.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ descargado: feature_importance_modelo_b.png


23. c

Genera y descarga tres visualizaciones comparativas:
baseline vs XGBoost para Modelo A, baseline vs XGBoost para Modelo B
ordenado por F1 ascendente, y la distribución de urgencia por tipo
de equipo en barras apiladas al 100%.

In [68]:
# ── CELDA 23c — BASELINE vs XGBOOST + DISTRIBUCIÓN PERIODO_FALLO ──
import matplotlib.pyplot as plt
import numpy as np
import os
from sklearn.metrics import f1_score
from google.colab import files

os.makedirs("/content/artefactos", exist_ok=True)
plt.style.use("dark_background")
BG    = "#0d1117"
ancho = 0.35

def exportar_y_descargar(fig, path):
    plt.savefig(path, dpi=150, bbox_inches="tight", facecolor=BG)
    plt.close()
    files.download(path)
    print("✓ descargado: {}".format(path.split("/")[-1]))

# ══════════════════════════════════════════════════════════════
# 1. BASELINE vs XGBOOST — MODELO A
# ══════════════════════════════════════════════════════════════
f1_dummy_a_clase = f1_score(y_a_test,
                             np.full_like(y_a_test, np.bincount(y_a_test).argmax()),
                             average=None)

x = np.arange(len(clases_a))

fig, ax = plt.subplots(figsize=(9, 5))
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

bars1 = ax.bar(x - ancho/2, f1_dummy_a_clase, ancho,
               label="Baseline (DummyClassifier)",
               color="#555555", linewidth=0)
bars2 = ax.bar(x + ancho/2, f1_a_clase, ancho,
               label="XGBoost + sample_weight",
               color=["#E24B4A", "#378ADD", "#1D9E75"], linewidth=0)

ax.axhline(f1_dummy_a, color="#888888", linewidth=1.2, linestyle="--",
           label="Macro baseline: {:.3f}".format(f1_dummy_a))
ax.axhline(f1_a, color="#ffffff", linewidth=1.2, linestyle="--",
           label="Macro XGBoost: {:.3f}".format(f1_a))

for bar in bars1:
    h = bar.get_height()
    if h > 0.01:
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.02,
                "{:.3f}".format(h),
                ha="center", va="bottom", color="#aaaaaa", fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            "{:.3f}".format(bar.get_height()),
            ha="center", va="bottom", color="#e0ddd6", fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(clases_a, color="#e0ddd6", fontsize=11)
ax.set_ylabel("F1-score", color="#e0ddd6")
ax.set_ylim(0, 1.15)
ax.set_title("Baseline vs XGBoost — Modelo A (Urgencia)",
             color="#e0ddd6", fontsize=13, pad=12)
ax.tick_params(colors="#e0ddd6")
ax.spines[:].set_color("#333333")
ax.legend(facecolor="#1a1a1a", edgecolor="#333333",
          labelcolor="#e0ddd6", fontsize=9)
plt.tight_layout()
exportar_y_descargar(fig, "/content/artefactos/baseline_vs_xgboost_modelo_a.png")

# ══════════════════════════════════════════════════════════════
# 2. BASELINE vs XGBOOST — MODELO B (ordenado por F1 XGBoost)
# ══════════════════════════════════════════════════════════════
f1_dummy_b_clase = f1_score(y_b_test,
                             np.full_like(y_b_test, np.bincount(y_b_test).argmax()),
                             average=None)

orden_b    = np.argsort(f1_b_clase)
clases_ord = [clases_b[i] for i in orden_b]
f1_xgb_ord = f1_b_clase[orden_b]
f1_dum_ord = f1_dummy_b_clase[orden_b]

n_clases_b = len(clases_b)
x_b        = np.arange(n_clases_b)

COLORES_B_BARRAS = [
    "#534AB7", "#7F77DD", "#AFA9EC", "#378ADD", "#1D9E75",
    "#EF9F27", "#E24B4A", "#E8836F", "#5DB8A0", "#A0C4FF",
][:n_clases_b]

fig, ax = plt.subplots(figsize=(13, 6))
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

bars1 = ax.bar(x_b - ancho/2, f1_dum_ord, ancho,
               label="Baseline (DummyClassifier)",
               color="#555555", linewidth=0)
bars2 = ax.bar(x_b + ancho/2, f1_xgb_ord, ancho,
               label="XGBoost + sample_weight",
               color=COLORES_B_BARRAS, linewidth=0)

ax.axhline(f1_dummy_b, color="#888888", linewidth=1.2, linestyle="--",
           label="Macro baseline: {:.3f}".format(f1_dummy_b))
ax.axhline(f1_b, color="#ffffff", linewidth=1.2, linestyle="--",
           label="Macro XGBoost: {:.3f}".format(f1_b))

for bar in bars1:
    h = bar.get_height()
    if h > 0.01:
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.02,
                "{:.3f}".format(h),
                ha="center", va="bottom", color="#aaaaaa", fontsize=7)
for bar, val in zip(bars2, f1_xgb_ord):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            "{:.3f}".format(val),
            ha="center", va="bottom", color="#e0ddd6", fontsize=7)

ax.set_xticks(x_b)
ax.set_xticklabels(clases_ord, color="#e0ddd6", fontsize=8,
                   rotation=30, ha="right")
ax.set_ylabel("F1-score", color="#e0ddd6")
ax.set_ylim(0, 1.20)
ax.set_title("Baseline vs XGBoost — Modelo B (Sistema afectado)",
             color="#e0ddd6", fontsize=13, pad=12)
ax.tick_params(colors="#e0ddd6")
ax.spines[:].set_color("#333333")
ax.legend(facecolor="#1a1a1a", edgecolor="#333333",
          labelcolor="#e0ddd6", fontsize=9)
plt.tight_layout()
exportar_y_descargar(fig, "/content/artefactos/baseline_vs_xgboost_modelo_b.png")

# ══════════════════════════════════════════════════════════════
# 3. DISTRIBUCIÓN PERIODO_FALLO POR TIPO DE EQUIPO
# ══════════════════════════════════════════════════════════════
if "tipo_corto" not in df_modelo.columns:
    df_modelo["tipo_corto"] = df_modelo["tipo_equipo"].map(NOMBRES_CORTOS)

orden_clases = ["Alta urgencia", "Programable", "Preventivo"]
COLORES_3    = ["#E24B4A", "#378ADD", "#1D9E75"]

tabla = (
    df_modelo
    .groupby(["tipo_corto", "periodo_fallo"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=orden_clases, fill_value=0)
)
tabla_pct = tabla.div(tabla.sum(axis=1), axis=0) * 100
tipos_ord  = tabla_pct["Alta urgencia"].sort_values(ascending=True).index.tolist()
tabla_pct  = tabla_pct.loc[tipos_ord]

fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

izq = np.zeros(len(tipos_ord))
for clase, color in zip(orden_clases, COLORES_3):
    vals = tabla_pct[clase].values
    bars = ax.barh(tipos_ord, vals, left=izq,
                   label=clase, color=color, linewidth=0)
    for bar, val in zip(bars, vals):
        if val > 5:
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_y() + bar.get_height()/2,
                    "{:.0f}%".format(val),
                    ha="center", va="center",
                    color="white", fontsize=9, fontweight="bold")
    izq += vals

ax.set_xlim(0, 100)
ax.set_xlabel("Proporción (%)", color="#e0ddd6")
ax.set_title("Distribución de urgencia por tipo de equipo",
             color="#e0ddd6", fontsize=13, pad=12)
ax.tick_params(colors="#e0ddd6")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: "{:.0f}%".format(v)))
ax.spines[:].set_color("#333333")
ax.legend(facecolor="#1a1a1a", edgecolor="#333333",
          labelcolor="#e0ddd6", fontsize=10,
          loc="lower right")
plt.tight_layout()
exportar_y_descargar(fig, "/content/artefactos/distribucion_periodo_por_equipo.png")

# ══════════════════════════════════════════════════════════════
# RE-DESCARGA — imágenes ya generadas en celda 23
# ══════════════════════════════════════════════════════════════
existentes = [
    "matriz_confusion_modelo_a.png",
    "matriz_confusion_modelo_b.png",
    "feature_importance_modelo_a.png",
    "feature_importance_modelo_b.png",
]
print()
print("Re-descargando imágenes de celda 23...")
for nombre in existentes:
    ruta = os.path.join("/content/artefactos", nombre)
    if os.path.exists(ruta):
        files.download(ruta)
        print("✓ {}".format(nombre))
    else:
        print("✗ {} — corre celda 23 primero".format(nombre))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ descargado: baseline_vs_xgboost_modelo_a.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

✓ feature_importance_modelo_a.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ feature_importance_modelo_b.png


# Bloque 5. Validación de los modelos.

Con ambos modelos entrenados, este bloque evalúa su capacidad de
generalización y justifica las decisiones metodológicas tomadas
durante el entrenamiento.

24. Validación cruzada y robustez. Se aplica validación cruzada
estratificada de 5 folds sobre el dataset completo para verificar que
los resultados no dependen del split aleatorio. Adicionalmente se evalúa
la robustez de ambos modelos inyectando ruido gaussiano progresivo
(5%, 10%, 20% y 30% de la desviación estándar) sobre las features
numéricas, midiendo la caída del F1-macro bajo condiciones de
perturbación controlada.

In [69]:
# ── CELDA 24 — VALIDACIÓN CRUZADA Y ROBUSTEZ — AMBOS MODELOS ──
from sklearn.model_selection import StratifiedKFold, cross_val_score
import numpy as np

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ══════════════════════════════════════════════════════════════
# MODELO A
# ══════════════════════════════════════════════════════════════

scores_a = cross_val_score(
    modelo_a, X_a, y_a,
    cv=skf, scoring="f1_macro", n_jobs=-1,
)

print("=" * 55)
print("  Validación cruzada 5-fold — Modelo A")
print("=" * 55)
print("  F1-macro por fold:")
for i, s in enumerate(scores_a):
    barra = "█" * int(s * 30)
    print("    Fold {}  {:.3f}  {}".format(i+1, s, barra))
print()
print("  F1-macro CV medio : {:.3f}".format(scores_a.mean()))
print("  Desviación std    : {:.3f}".format(scores_a.std()))
print("  F1-macro test     : {:.3f}".format(f1_a))
print("  Diferencia CV-test: {:.3f}".format(abs(scores_a.mean() - f1_a)))
if abs(scores_a.mean() - f1_a) < 0.05:
    print("  ✓ Diferencia < 0.05 → modelo no memorizó el split")
else:
    print("  ⚠ Diferencia > 0.05 → revisar posible sobreajuste al split")

print()
print("=" * 55)
print("  Robustez — Modelo A — ruido gaussiano")
print("  Solo features numéricas perturbadas")
print("=" * 55)
print("  F1-macro sin ruido : {:.3f}".format(f1_a))
print()

idx_num_a = [FEATURES_A.index(f) for f in FEATURES_A if f not in FEATURES_A_CAT]

resultados_ruido_a = []
for nivel in [0.05, 0.10, 0.20, 0.30]:
    np.random.seed(42)
    X_ruidoso = X_a_test.copy().astype(float)
    for idx in idx_num_a:
        std   = X_ruidoso[:, idx].std()
        ruido = np.random.normal(0, nivel * std, size=X_ruidoso.shape[0])
        X_ruidoso[:, idx] += ruido
    f1_r  = f1_score(y_a_test, modelo_a.predict(X_ruidoso), average="macro")
    caida = f1_a - f1_r
    resultados_ruido_a.append((nivel, f1_r, caida))
    barra = "█" * int(f1_r * 30)
    print("  Ruido {:>3.0f}% std : F1 = {:.3f}  caída: {:.3f}  {}".format(
        nivel*100, f1_r, caida, barra))

print()
f1_min_a   = min(r[1] for r in resultados_ruido_a)
caida_max_a = max(r[2] for r in resultados_ruido_a)
if caida_max_a < 0.05:
    print("  ✓ Caída máxima < 0.05 → modelo muy robusto")
elif caida_max_a < 0.10:
    print("  ⚡ Caída máxima < 0.10 → robustez aceptable")
else:
    print("  ⚠ Caída máxima > 0.10 → modelo sensible a perturbaciones")
print("  Piso de rendimiento con ruido: {:.3f}".format(f1_min_a))

# ══════════════════════════════════════════════════════════════
# MODELO B
# ══════════════════════════════════════════════════════════════

print()
scores_b = cross_val_score(
    modelo_b, X_b, y_b,
    cv=skf, scoring="f1_macro", n_jobs=-1,
)

print("=" * 55)
print("  Validación cruzada 5-fold — Modelo B")
print("=" * 55)
print("  F1-macro por fold:")
for i, s in enumerate(scores_b):
    barra = "█" * int(s * 30)
    print("    Fold {}  {:.3f}  {}".format(i+1, s, barra))
print()
print("  F1-macro CV medio : {:.3f}".format(scores_b.mean()))
print("  Desviación std    : {:.3f}".format(scores_b.std()))
print("  F1-macro test     : {:.3f}".format(f1_b))
print("  Diferencia CV-test: {:.3f}".format(abs(scores_b.mean() - f1_b)))
if abs(scores_b.mean() - f1_b) < 0.05:
    print("  ✓ Diferencia < 0.05 → modelo no memorizó el split")
else:
    print("  ⚠ Diferencia > 0.05 → revisar posible sobreajuste al split")

print()
print("=" * 55)
print("  Robustez — Modelo B — ruido gaussiano")
print("  Solo features numéricas perturbadas")
print("=" * 55)
print("  F1-macro sin ruido : {:.3f}".format(f1_b))
print()

idx_num_b = [FEATURES_B.index(f) for f in FEATURES_B if f not in FEATURES_B_CAT]

resultados_ruido_b = []
for nivel in [0.05, 0.10, 0.20, 0.30]:
    np.random.seed(42)
    X_ruidoso = X_b_test.copy().astype(float)
    for idx in idx_num_b:
        std   = X_ruidoso[:, idx].std()
        ruido = np.random.normal(0, nivel * std, size=X_ruidoso.shape[0])
        X_ruidoso[:, idx] += ruido
    f1_r  = f1_score(y_b_test, modelo_b.predict(X_ruidoso), average="macro")
    caida = f1_b - f1_r
    resultados_ruido_b.append((nivel, f1_r, caida))
    barra = "█" * int(f1_r * 30)
    print("  Ruido {:>3.0f}% std : F1 = {:.3f}  caída: {:.3f}  {}".format(
        nivel*100, f1_r, caida, barra))

print()
f1_min_b    = min(r[1] for r in resultados_ruido_b)
caida_max_b = max(r[2] for r in resultados_ruido_b)
if caida_max_b < 0.05:
    print("  ✓ Caída máxima < 0.05 → modelo muy robusto")
elif caida_max_b < 0.10:
    print("  ⚡ Caída máxima < 0.10 → robustez aceptable")
else:
    print("  ⚠ Caída máxima > 0.10 → modelo sensible a perturbaciones")
print("  Piso de rendimiento con ruido: {:.3f}".format(f1_min_b))

# ── Resumen comparativo ────────────────────────────────────────
print()
print("=" * 55)
print("  Resumen comparativo")
print("=" * 55)
print("  {:<12} {:>12} {:>12} {:>12} {:>12}".format(
    "Modelo", "F1 test", "F1 CV medio", "Std CV", "Piso ruido"))
print("  " + "-" * 52)
print("  {:<12} {:>12.3f} {:>12.3f} {:>12.3f} {:>12.3f}".format(
    "Modelo A", f1_a, scores_a.mean(), scores_a.std(), f1_min_a))
print("  {:<12} {:>12.3f} {:>12.3f} {:>12.3f} {:>12.3f}".format(
    "Modelo B", f1_b, scores_b.mean(), scores_b.std(), f1_min_b))

  Validación cruzada 5-fold — Modelo A
  F1-macro por fold:
    Fold 1  0.850  █████████████████████████
    Fold 2  0.844  █████████████████████████
    Fold 3  0.825  ████████████████████████
    Fold 4  0.860  █████████████████████████
    Fold 5  0.848  █████████████████████████

  F1-macro CV medio : 0.845
  Desviación std    : 0.012
  F1-macro test     : 0.872
  Diferencia CV-test: 0.027
  ✓ Diferencia < 0.05 → modelo no memorizó el split

  Robustez — Modelo A — ruido gaussiano
  Solo features numéricas perturbadas
  F1-macro sin ruido : 0.872

  Ruido   5% std : F1 = 0.701  caída: 0.171  █████████████████████
  Ruido  10% std : F1 = 0.697  caída: 0.175  ████████████████████
  Ruido  20% std : F1 = 0.687  caída: 0.185  ████████████████████
  Ruido  30% std : F1 = 0.663  caída: 0.209  ███████████████████

  ⚠ Caída máxima > 0.10 → modelo sensible a perturbaciones
  Piso de rendimiento con ruido: 0.663

  Validación cruzada 5-fold — Modelo B
  F1-macro por fold:
    Fold 1  0.931 

25. Experimento SMOTE. Se evalúan dos estrategias de
sobremuestreo sintético para tratar el desbalance de clases:
SMOTE total (iguala todas las clases a la mayoritaria) y SMOTE parcial
(sube las clases minoritarias al 60% de la mayoritaria). Ambas variantes
se comparan contra el modelo original en F1 test, gap de sobreajuste
y piso de robustez con ruido, justificando la decisión de usar
compute_sample_weight("balanced") en lugar de datos sintéticos.

In [70]:
# ── CELDA 25 — EXPERIMENTO SMOTE — JUSTIFICACIÓN DE NO USO ────
try:
    from imblearn.over_sampling import SMOTE
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "imbalanced-learn", "-q"])
    from imblearn.over_sampling import SMOTE

from sklearn.metrics import f1_score, classification_report
from xgboost import XGBClassifier
import numpy as np

# ══════════════════════════════════════════════════════════════
# FUNCIÓN AUXILIAR — entrenar y evaluar con SMOTE
# ══════════════════════════════════════════════════════════════
def evaluar_smote(X_train, y_train, X_test, y_test,
                  idx_num, clases, smote, nombre_modelo):
    X_sm, y_sm = smote.fit_resample(X_train, y_train)

    modelo_sm = XGBClassifier(
        n_estimators=200, max_depth=3, learning_rate=0.05,
        subsample=0.7, colsample_bytree=0.7, min_child_weight=5,
        reg_alpha=0.1, reg_lambda=2.0,
        eval_metric="mlogloss", random_state=42, n_jobs=-1,
    )
    modelo_sm.fit(X_sm, y_sm)

    y_pred       = modelo_sm.predict(X_test)
    y_pred_train = modelo_sm.predict(X_sm)

    f1_test  = f1_score(y_test, y_pred,       average="macro")
    f1_train = f1_score(y_sm,   y_pred_train, average="macro")
    f1_clase = f1_score(y_test, y_pred,       average=None)
    gap      = f1_train - f1_test

    pisos = []
    for nivel in [0.05, 0.10, 0.20, 0.30]:
        np.random.seed(42)
        X_r = X_test.copy().astype(float)
        for idx in idx_num:
            std = X_r[:, idx].std()
            X_r[:, idx] += np.random.normal(0, nivel * std, size=X_r.shape[0])
        pisos.append(f1_score(y_test, modelo_sm.predict(X_r), average="macro"))

    return {
        "f1_test"  : f1_test,
        "f1_train" : f1_train,
        "gap"      : gap,
        "f1_clase" : f1_clase,
        "f1_min"   : min(pisos),
        "y_pred"   : y_pred,
    }

# ══════════════════════════════════════════════════════════════
# SMOTE TOTAL — iguala todas las clases a la mayoritaria
# ══════════════════════════════════════════════════════════════
smote_total = SMOTE(random_state=42, k_neighbors=5)

res_a_total = evaluar_smote(
    X_a_train, y_a_train, X_a_test, y_a_test,
    idx_num_a, clases_a, smote_total, "Modelo A"
)
res_b_total = evaluar_smote(
    X_b_train, y_b_train, X_b_test, y_b_test,
    idx_num_b, clases_b, smote_total, "Modelo B"
)

# ══════════════════════════════════════════════════════════════
# SMOTE PARCIAL — sube clases pequeñas al 60% de la mayoritaria
# ══════════════════════════════════════════════════════════════
def estrategia_parcial(y_train, factor=0.60):
    clases_idx, conteos = np.unique(y_train, return_counts=True)
    objetivo = int(max(conteos) * factor)
    return {
        int(c): max(n, objetivo)
        for c, n in zip(clases_idx, conteos)
    }

smote_parcial_a = SMOTE(
    sampling_strategy=estrategia_parcial(y_a_train, factor=0.60),
    random_state=42, k_neighbors=5,
)
smote_parcial_b = SMOTE(
    sampling_strategy=estrategia_parcial(y_b_train, factor=0.60),
    random_state=42, k_neighbors=5,
)

res_a_parcial = evaluar_smote(
    X_a_train, y_a_train, X_a_test, y_a_test,
    idx_num_a, clases_a, smote_parcial_a, "Modelo A"
)
res_b_parcial = evaluar_smote(
    X_b_train, y_b_train, X_b_test, y_b_test,
    idx_num_b, clases_b, smote_parcial_b, "Modelo B"
)

# ══════════════════════════════════════════════════════════════
# TABLA COMPARATIVA — MODELO A
# ══════════════════════════════════════════════════════════════
print("=" * 68)
print("  COMPARATIVA SMOTE — MODELO A")
print("=" * 68)
print("  {:<28} {:>10} {:>10} {:>12}".format(
    "Versión", "F1 test", "Gap", "Piso ruido"))
print("  " + "-" * 62)
print("  {:<28} {:>10.3f} {:>10.3f} {:>12.3f}".format(
    "Original (sample_weight)", f1_a, f1_a_train - f1_a, f1_min_a))
print("  {:<28} {:>10.3f} {:>10.3f} {:>12.3f}".format(
    "SMOTE total", res_a_total["f1_test"], res_a_total["gap"], res_a_total["f1_min"]))
print("  {:<28} {:>10.3f} {:>10.3f} {:>12.3f}".format(
    "SMOTE parcial (60%)", res_a_parcial["f1_test"], res_a_parcial["gap"], res_a_parcial["f1_min"]))
print()
print("  F1 por clase — Modelo A:")
print("  {:<22} {:>10} {:>12} {:>14}".format(
    "Clase", "Original", "SMOTE total", "SMOTE parcial"))
print("  " + "-" * 62)
for nombre, v_orig, v_tot, v_par in zip(
        clases_a, f1_a_clase,
        res_a_total["f1_clase"], res_a_parcial["f1_clase"]):
    print("  {:<22} {:>10.3f} {:>12.3f} {:>14.3f}".format(
        nombre, v_orig, v_tot, v_par))

# ══════════════════════════════════════════════════════════════
# TABLA COMPARATIVA — MODELO B
# ══════════════════════════════════════════════════════════════
print()
print("=" * 68)
print("  COMPARATIVA SMOTE — MODELO B")
print("=" * 68)
print("  {:<28} {:>10} {:>10} {:>12}".format(
    "Versión", "F1 test", "Gap", "Piso ruido"))
print("  " + "-" * 62)
print("  {:<28} {:>10.3f} {:>10.3f} {:>12.3f}".format(
    "Original (sample_weight)", f1_b, f1_b_train - f1_b, f1_min_b))
print("  {:<28} {:>10.3f} {:>10.3f} {:>12.3f}".format(
    "SMOTE total", res_b_total["f1_test"], res_b_total["gap"], res_b_total["f1_min"]))
print("  {:<28} {:>10.3f} {:>10.3f} {:>12.3f}".format(
    "SMOTE parcial (60%)", res_b_parcial["f1_test"], res_b_parcial["gap"], res_b_parcial["f1_min"]))
print()
print("  F1 por clase — Modelo B:")
print("  {:<35} {:>10} {:>12} {:>14}".format(
    "Clase", "Original", "SMOTE total", "SMOTE parcial"))
print("  " + "-" * 73)
for nombre, v_orig, v_tot, v_par in zip(
        clases_b, f1_b_clase,
        res_b_total["f1_clase"], res_b_parcial["f1_clase"]):
    print("  {:<35} {:>10.3f} {:>12.3f} {:>14.3f}".format(
        str(nombre), v_orig, v_tot, v_par))

# ══════════════════════════════════════════════════════════════
# DECISIÓN FINAL
# ══════════════════════════════════════════════════════════════
print()
print("=" * 68)
print("  DECISIÓN — JUSTIFICACIÓN DE NO USO DE SMOTE")
print("=" * 68)
print()
print("  Se evaluaron dos variantes de SMOTE:")
print("  · SMOTE total   : iguala todas las clases a la mayoritaria")
print("  · SMOTE parcial : sube clases pequeñas al 60% de la mayoritaria")
print()
print("  Modelo A — ninguna variante supera al original:")
d_tot_a = res_a_total["f1_test"]   - f1_a
d_par_a = res_a_parcial["f1_test"] - f1_a
print("    SMOTE total   → F1 {:+.3f}  gap {:+.3f}".format(
    d_tot_a, res_a_total["gap"]   - (f1_a_train - f1_a)))
print("    SMOTE parcial → F1 {:+.3f}  gap {:+.3f}".format(
    d_par_a, res_a_parcial["gap"] - (f1_a_train - f1_a)))
print()
print("  Modelo B — ninguna variante supera al original:")
d_tot_b = res_b_total["f1_test"]   - f1_b
d_par_b = res_b_parcial["f1_test"] - f1_b
print("    SMOTE total   → F1 {:+.3f}  gap {:+.3f}".format(
    d_tot_b, res_b_total["gap"]   - (f1_b_train - f1_b)))
print("    SMOTE parcial → F1 {:+.3f}  gap {:+.3f}".format(
    d_par_b, res_b_parcial["gap"] - (f1_b_train - f1_b)))
print()
print("  CONCLUSIÓN:")
print("  Ambas variantes de SMOTE aumentan el gap de sobreajuste")
print("  sin mejorar el F1 en test. La interpolación sintética")
print("  entre registros de mantenimiento no captura patrones")
print("  reales de falla — cada orden es un evento único.")
print("  compute_sample_weight('balanced') ajusta la penalización")
print("  durante el entrenamiento sin alterar los datos originales,")
print("  preservando la distribución real de fallas 2020-2025.")
print("  DECISIÓN FINAL: se descarta SMOTE en ambos modelos.")

  COMPARATIVA SMOTE — MODELO A
  Versión                         F1 test        Gap   Piso ruido
  --------------------------------------------------------------
  Original (sample_weight)          0.872      0.003        0.663
  SMOTE total                       0.872      0.025        0.695
  SMOTE parcial (60%)               0.872      0.022        0.657

  F1 por clase — Modelo A:
  Clase                    Original  SMOTE total  SMOTE parcial
  --------------------------------------------------------------
  Alta urgencia               0.829        0.827          0.824
  Programable                 0.825        0.827          0.830
  Preventivo                  0.962        0.962          0.962

  COMPARATIVA SMOTE — MODELO B
  Versión                         F1 test        Gap   Piso ruido
  --------------------------------------------------------------
  Original (sample_weight)          0.910      0.046        0.853
  SMOTE total                       0.901      0.061        0.

# Bloque 6. Construcción de protocolos de mantenimiento

Con los modelos validados, este bloque cruza las predicciones de ambos
clasificadores con el historial real de intervenciones para construir
la base empírica de los protocolos de mantenimiento predictivo.

27. Check list por tipo de equipo. A partir de la tabla maestra,
reclasifica cada sistema en una única categoría de urgencia dominante
(la de mayor probabilidad histórica) y reemplaza el semáforo provisional
por uno basado en el F1 real del Modelo B por clase, que es la medida
más rigurosa de confianza disponible. Para cada combinación
tipo_equipo × urgencia × sistema genera una ficha con la acción
histórica más frecuente, el detalle de intervención, el componente
a verificar, el tiempo estimado de intervención basado en el promedio
histórico de horas de parada, y el nivel de confianza del modelo.
Las fichas se ordenan de mayor a menor riesgo dentro de cada equipo
y se exportan en checklist_base.csv.

In [71]:
# ─────────────────────────────────────────────────────────────
# CELDA 27 — CHECK LIST POR TIPO DE EQUIPO (VERSIÓN FINAL)
# ─────────────────────────────────────────────────────────────

import pandas as pd
import numpy as np
from IPython.display import display, HTML

# ── 1. Calcular métricas desde df_modelo ─────────────────────
if "tipo_corto" not in df_modelo.columns:
    df_modelo["tipo_corto"] = df_modelo["tipo_equipo"].map(NOMBRES_CORTOS)

# Limpiar componente_cambiado
INVALIDOS = ["sin dato", "no aplica", "nan", "NaN", "-", "", "n/a", "na", "none"]
df_comp = df_modelo.copy()
df_comp["componente_cambiado"] = df_comp["componente_cambiado"].astype(str).str.strip()
df_comp["_comp_valido"] = ~df_comp["componente_cambiado"].isin(INVALIDOS)

# Para cada tipo × sistema: encontrar el componente_cambiado más frecuente
top_comp_raw = (
    df_comp[df_comp["_comp_valido"]]
    .groupby(["tipo_corto", "sistema_general_target", "componente_cambiado"])
    .agg(
        n_comp        = ("horas_de_parada_num", "count"),
        horas_prom_c  = ("horas_de_parada_num", "mean"),
        horas_max_c   = ("horas_de_parada_num", "max"),
    )
    .reset_index()
    .sort_values(["tipo_corto", "sistema_general_target", "n_comp"], ascending=[True, True, False])
)

# Quedarnos solo con el top 1 componente por tipo × sistema
top_comp = (
    top_comp_raw
    .groupby(["tipo_corto", "sistema_general_target"])
    .first()
    .reset_index()
    .rename(columns={
        "sistema_general_target": "sistema_general",
        "componente_cambiado":    "componente_top",
        "horas_prom_c":           "horas_parada_prom",
        "horas_max_c":            "horas_parada_max",
    })
)[["tipo_corto", "sistema_general", "componente_top", "horas_parada_prom", "horas_parada_max"]]

# Tasa de cambio de componente (sobre todos los registros del sistema)
tasa_cambio = (
    df_modelo.groupby(["tipo_corto", "sistema_general_target"])["hubo_cambio_componente"]
    .mean()
    .reset_index()
    .rename(columns={
        "sistema_general_target": "sistema_general",
        "hubo_cambio_componente": "tasa_cambio_componente"
    })
)

# Top acción
top_accion = (
    df_modelo.groupby(["tipo_corto", "sistema_general_target"])["tipo_solucion"]
    .agg(lambda x: x.value_counts().idxmax() if len(x) > 0 else "sin dato")
    .reset_index()
    .rename(columns={
        "sistema_general_target": "sistema_general",
        "tipo_solucion":          "tipo_solucion_top"
    })
)

# Unir todo
metricas = (
    top_comp
    .merge(tasa_cambio, on=["tipo_corto", "sistema_general"], how="left")
    .merge(top_accion,  on=["tipo_corto", "sistema_general"], how="left")
)

# ── 2. Aplicar semáforo F1 a df_clasificado ──────────────────
df_clasificado[["semaforo", "f1"]] = df_clasificado.apply(
    lambda r: pd.Series(semaforo_f1(r["sistema_general"])), axis=1
)

# ── 3. Unir métricas al checklist ────────────────────────────
checklist = df_clasificado.merge(metricas, on=["tipo_corto", "sistema_general"], how="left")

checklist["horas_parada_prom"]      = checklist["horas_parada_prom"].fillna(0).round(1)
checklist["horas_parada_max"]       = checklist["horas_parada_max"].fillna(0).round(0).astype(int)
checklist["tasa_cambio_componente"] = checklist["tasa_cambio_componente"].fillna(0)
checklist["componente_top"]         = checklist["componente_top"].fillna("sin dato")
checklist["tipo_solucion_top"]      = checklist["tipo_solucion_top"].fillna("sin dato")

checklist.to_csv("/content/checklist_base.csv", index=False)
print("✅ checklist_base.csv guardada")
print(f"   {len(checklist)} sistemas · {checklist['tipo_corto'].nunique()} equipos")

# ── 4. Helpers visuales ───────────────────────────────────────
def bar_pct(pct, color, ancho=55):
    w = min(ancho, int(pct * ancho / 100))
    return (
        f"<div style='display:flex;align-items:center;gap:5px'>"
        f"<div style='width:{ancho}px;background:#1e1e1e;border-radius:2px;height:6px'>"
        f"<div style='width:{w}px;background:{color};border-radius:2px;height:6px'></div>"
        f"</div>"
        f"<span style='color:#aaa;font-size:11px;min-width:34px'>{pct:.0f}%</span>"
        f"</div>"
    )

ICONOS_CLAS = {"Alta urgencia": "⚡", "Programable": "📅", "Preventivo": "✅"}
COLORES_CLAS = {
    "Alta urgencia": {"text": "#E24B4A", "bg": "#2a1010"},
    "Programable":   {"text": "#378ADD", "bg": "#0d1a2a"},
    "Preventivo":    {"text": "#1D9E75", "bg": "#0d2a1a"},
}
COLORES_SEM = {
    "VERDE":    {"bg": "#0d2b1a", "borde": "#1D9E75"},
    "AMARILLO": {"bg": "#2b2500", "borde": "#EF9F27"},
    "ROJO":     {"bg": "#2b0d0d", "borde": "#E24B4A"},
}
ACCION_LABEL = {
    "Cambio de componente"                      : "🔧 Cambiar componente",
    "Ajuste mecánico básico"                    : "🔩 Ajuste mecánico",
    "Ajuste funcionamiento de componente"       : "🔩 Ajuste de componente",
    "Reinicio del equipo"                       : "🔄 Reinicio del equipo",
    "Recarga de software"                       : "💾 Recarga de software",
    "Actualización de software"                 : "💾 Actualizar software",
    "Borrado de archivos"                       : "💾 Limpieza software",
    "Calibración detector"                      : "📐 Calibrar detector",
    "Calibración"                               : "📐 Calibración",
    "Configuracion de parametros"               : "⚙️ Configurar parámetros",
    "Configuracion de protocolos"               : "⚙️ Configurar protocolos",
    "Ajuste de conexión con plataforma clínica" : "🌐 Ajuste red/PACS",
    "Conexion de red electrica"                 : "⚡ Revisión eléctrica",
    "Ajuste de receptáculo"                     : "🔌 Revisión eléctrica FVL",
    "Cambio IP"                                 : "🌐 Cambio IP",
    "Conexión de red"                           : "🌐 Revisión red TI",
    "Verificación nodo de red"                  : "🌐 Verificar red TI",
    "Asesoria / capacitacion"                   : "👤 Capacitación usuario",
    "Correcion por mal uso"                     : "👤 Corrección uso",
    "Evento esporádico"                         : "👁️ Monitoreo",
    "Reparación (sin cambio)"                   : "🔧 Reparación sin cambio",
}

# ── 5. Visualización HTML ─────────────────────────────────────
estilos = (
    "<style>"
    ".cl{font-family:Arial,sans-serif;color:#e0ddd6;padding:8px 0}"
    ".cl-tipo{margin:24px 0 0;padding:12px 18px;border-radius:8px 8px 0 0;"
    "font-size:14px;font-weight:700;background:#1a1a1a;border-left:5px solid #555;"
    "display:flex;align-items:center;gap:12px}"
    ".cl-sem-sum{margin-left:auto;display:flex;gap:10px;font-size:11px}"
    ".cl-table{width:100%;border-collapse:collapse;border:1px solid #2a2a2a;"
    "border-top:none;margin-bottom:4px}"
    ".cl-table th{background:#141414;padding:8px 10px;font-size:10px;color:#555;"
    "font-weight:700;text-align:left;border-bottom:1px solid #2a2a2a;"
    "text-transform:uppercase;letter-spacing:.06em;white-space:nowrap}"
    ".cl-table td{padding:9px 10px;font-size:12px;border-bottom:1px solid #181818;"
    "background:#0d0d0d;vertical-align:middle}"
    ".cl-table tr:hover td{background:#131313}"
    ".cls-b{padding:3px 10px;border-radius:99px;font-size:11px;font-weight:700;"
    "white-space:nowrap}"
    ".sem-d{width:9px;height:9px;border-radius:50%;display:inline-block;margin-right:5px}"
    ".chk{width:14px;height:14px;border:2px solid #333;border-radius:3px;"
    "display:inline-block;margin-right:5px;vertical-align:middle;flex-shrink:0}"
    ".comp-cell{display:flex;align-items:center;gap:4px;font-size:11px;color:#999}"
    ".sep-urg td{background:#111 !important;height:2px;padding:0 !important;"
    "border-bottom:1px solid #2a2a2a !important}"
    "</style>"
)

html = estilos + "<div class='cl'>"
orden_riesgo = {"Alta urgencia": 0, "Programable": 1, "Preventivo": 2}

for tipo in sorted(checklist["tipo_corto"].unique()):
    sub = checklist[checklist["tipo_corto"] == tipo].copy()
    sub["_orden"] = sub["clasificacion"].map(orden_riesgo).fillna(3)
    sub = sub.sort_values(["_orden", "p_alta"], ascending=[True, False])

    n_tot      = int(sub["total"].sum())
    n_verde    = (sub["semaforo"] == "VERDE").sum()
    n_amarillo = (sub["semaforo"] == "AMARILLO").sum()
    n_rojo     = (sub["semaforo"] == "ROJO").sum()
    borde      = "#E24B4A" if (sub["clasificacion"] == "Alta urgencia").any() else "#378ADD"

    html += (
        f"<div class='cl-tipo' style='border-left-color:{borde}'>"
        f"📋 Check List — <strong>{tipo}</strong>"
        f"<span style='font-weight:400;color:#555;font-size:11px'>"
        f"&nbsp;·&nbsp;{n_tot} registros históricos</span>"
        f"<div class='cl-sem-sum'>"
        f"<span style='color:#1D9E75'>🟢 {n_verde}</span>"
        f"<span style='color:#EF9F27'>🟡 {n_amarillo}</span>"
        f"<span style='color:#E24B4A'>🔴 {n_rojo}</span>"
        f"</div></div>"
        "<table class='cl-table'><thead><tr>"
        "<th style='width:20px'></th>"
        "<th>Sistema afectado</th>"
        "<th>Clasificación</th>"
        "<th>%Alta</th>"
        "<th>%Prog</th>"
        "<th>%Prev</th>"
        "<th style='text-align:center'>N Total</th>"
        "<th>H. Parada Prom</th>"
        "<th>H. Parada Máx</th>"
        "<th style='text-align:center'>% Cambio Comp.</th>"
        "<th>Componente Top</th>"
        "<th>Acción principal</th>"
        "<th style='text-align:center'>F1 modelo B</th>"
        "<th>Confianza</th>"
        "</tr></thead><tbody>"
    )

    prev_cls = None
    for _, row in sub.iterrows():
        cls = row["clasificacion"]
        cc  = COLORES_CLAS.get(cls, COLORES_CLAS["Programable"])
        cs  = COLORES_SEM.get(str(row.get("semaforo", "ROJO")), COLORES_SEM["ROJO"])

        if prev_cls is not None and cls != prev_cls:
            html += "<tr class='sep-urg'><td colspan='14'></td></tr>"
        prev_cls = cls

        badge = (
            f"<span class='cls-b' style='color:{cc['text']};background:{cc['bg']}'>"
            f"{ICONOS_CLAS.get(cls,'')} {cls}</span>"
        )

        accion_raw = str(row.get("tipo_solucion_top", "sin dato"))
        accion     = ACCION_LABEL.get(accion_raw, f"🔧 {accion_raw}")
        comp       = str(row.get("componente_top", "sin dato"))
        h_prom     = f"{row['horas_parada_prom']:.1f} h" if row["horas_parada_prom"] > 0 else "< 1 h"
        h_max      = f"{int(row['horas_parada_max'])} h"  if row["horas_parada_max"]  > 0 else "< 1 h"
        tasa_c     = row.get("tasa_cambio_componente", 0)
        tasa_str   = f"{tasa_c:.0%}" if pd.notna(tasa_c) else "—"
        f1_val     = f"{row['f1']:.3f}" if pd.notna(row.get("f1")) else "—"
        sem        = str(row.get("semaforo", "ROJO"))

        html += (
            f"<tr>"
            f"<td><span class='chk'></span></td>"
            f"<td style='color:#e0ddd6;font-weight:600'>{row['sistema_general']}</td>"
            f"<td>{badge}</td>"
            f"<td>{bar_pct(row['p_alta'],        '#E24B4A')}</td>"
            f"<td>{bar_pct(row['p_programable'], '#378ADD')}</td>"
            f"<td>{bar_pct(row['p_preventivo'],  '#1D9E75')}</td>"
            f"<td style='color:#aaa;text-align:center;font-weight:600'>{int(row['total'])}</td>"
            f"<td style='color:#1D9E75;font-weight:600'>{h_prom}</td>"
            f"<td style='color:#EF9F27;font-size:11px'>{h_max}</td>"
            f"<td style='color:#aaa;text-align:center'>{tasa_str}</td>"
            f"<td><div class='comp-cell'><span class='chk'></span>{comp}</div></td>"
            f"<td style='color:#e0ddd6'>{accion}</td>"
            f"<td style='text-align:center;color:#aaa;font-weight:600'>{f1_val}</td>"
            f"<td style='background:{cs['bg']};border-left:3px solid {cs['borde']}'>"
            f"<div style='display:flex;align-items:center'>"
            f"<div class='sem-d' style='background:{cs['borde']}'></div>"
            f"<span style='color:{cs['borde']};font-size:10px;font-weight:700'>{sem}</span>"
            f"</div></td>"
            f"</tr>"
        )

    html += "</tbody></table>"

html += "</div>"

with open("/content/checklist_final.html", "w", encoding="utf-8") as f:
    f.write(html)

display(HTML(html))

# ── 6. Resumen consola ────────────────────────────────────────
print()
print("=" * 65)
print("  CHECK LIST — Resumen por tipo de equipo")
print("=" * 65)
for tipo in sorted(checklist["tipo_corto"].unique()):
    sub  = checklist[checklist["tipo_corto"] == tipo]
    n_a  = (sub["clasificacion"] == "Alta urgencia").sum()
    n_p  = (sub["clasificacion"] == "Programable").sum()
    n_v  = (sub["clasificacion"] == "Preventivo").sum()
    n_v2 = (sub["semaforo"] == "VERDE").sum()
    n_am = (sub["semaforo"] == "AMARILLO").sum()
    n_r  = (sub["semaforo"] == "ROJO").sum()
    print(f"  {tipo:<22}  ⚡{n_a} 📅{n_p} ✅{n_v}  |  🟢{n_v2} 🟡{n_am} 🔴{n_r}")

print()
print("  Archivos generados:")
print("    /content/checklist_base.csv")
print("    /content/checklist_final.html")

✅ checklist_base.csv guardada
   67 sistemas · 7 equipos



  CHECK LIST — Resumen por tipo de equipo
  Angiografía             ⚡2 📅7 ✅1  |  🟢9 🟡1 🔴0
  Arco C                  ⚡1 📅5 ✅4  |  🟢9 🟡1 🔴0
  Fluoro                  ⚡4 📅4 ✅1  |  🟢8 🟡1 🔴0
  Mamografía              ⚡0 📅9 ✅0  |  🟢8 🟡1 🔴0
  RX Fijo                 ⚡6 📅3 ✅0  |  🟢8 🟡1 🔴0
  RX Móvil                ⚡4 📅6 ✅0  |  🟢9 🟡1 🔴0
  TAC                     ⚡8 📅2 ✅0  |  🟢9 🟡1 🔴0

  Archivos generados:
    /content/checklist_base.csv
    /content/checklist_final.html
